# Content-based Kindle Recommendation — Sequential-protocol-aligned

This notebook compares content-based recommendation variants under the same chronological candidate-set protocol as `Kindle_SASRec_Sequential_Recommender.ipynb`.

The fixed protocol is:

- validation predicts `val.pos_idx` from the user's training history;
- testing predicts `test.pos_idx` from training history **plus the validation interaction**;
- every target is ranked against its same 100 fixed negatives;
- all model and hyperparameter selection uses validation `NDCG@10`;
- MRR is computed over all 101 candidates;
- popularity uses only training interaction counts;
- test evaluation appears only in the final locked-configuration cell.

The candidate-set metrics are not full-catalogue Top-10 metrics. Full-catalogue coverage, novelty, and diversity are reported separately on the same deterministic diagnostic-user sample used by the sequential notebook.


# Setup and Configuration

### Environment setup (local machine or Colab, either way)

Colab already has `numpy`/`pandas`/`scipy`/`scikit-learn`/`joblib` preinstalled, so this
cell is a no-op there -- but a fresh local Python environment (a new venv, a freshly
cloned repo on your own machine) usually doesn't have them yet. This cell checks each one
and `pip install`s only whatever is actually missing, so the rest of the notebook runs
the same way whether you're on Colab, your own laptop, or any other machine.

`sentence-transformers` (needed only for the SBERT variant) is deliberately **not**
included here -- it's checked separately, right before that section, since it also pulls
in `torch` (a large download) and is only needed if `CONFIG["run_sbert"]` is `True`.

In [2]:
import importlib
import subprocess
import sys

# import name -> pip package name (they differ for scikit-learn)
REQUIRED_CORE_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}

missing_packages = [
    pip_name
    for import_name, pip_name in REQUIRED_CORE_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print(f"Installing missing core packages: {missing_packages} ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )
    print("Done.")
else:
    print("All core packages (numpy/pandas/scipy/scikit-learn/joblib) already available.")


All core packages (numpy/pandas/scipy/scikit-learn/joblib) already available.


### Colab bootstrap (safe to run anywhere)

This cell is a no-op outside Colab -- `IN_COLAB` is False, Drive is never mounted, and
`COLAB_DATA_DIR_CANDIDATES` stays empty, so `CONFIG["data_dir_candidates"]` below falls
back to the original local-relative paths exactly as before. On Colab, it mounts Drive
and adds a couple of common Drive layouts as *additional* candidates -- `DATA_DIR` (in the
Data Loading section) still just picks whichever candidate path actually exists.

**Adjust `COLAB_DATA_DIR_CANDIDATES` below if your `processed_kindle/` folder lives
somewhere else in Drive** -- this is only a guess at a common layout, not a guarantee.

**Output files are also rooted in Drive on Colab** via `COLAB_OUTPUT_ROOT` -- every artifact this notebook writes (`candidate_set_results.csv`, catalogue diagnostics, the model-handoff files, the candidate-scores export, the SBERT cache) ends up under that folder instead of Colab's ephemeral `/content` disk, so nothing is lost if the runtime restarts or disconnects. Adjust `COLAB_OUTPUT_ROOT` below if you want a different Drive folder; off Colab, `OUTPUT_DIR`/`SHARED_OUTPUTS_DIR` are completely unchanged from before.

In [3]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Adjust these if your processed_kindle/ folder lives somewhere else in Drive.
    COLAB_DATA_DIR_CANDIDATES = (
        "/content/drive/MyDrive/processed_kindle/",
        "/content/drive/MyDrive/proj2/processed_kindle/",
    )
    # All output artifacts (results CSVs, model handoff files, candidate scores, the
    # SBERT cache) get rooted here instead of Colab's ephemeral /content disk, so they
    # survive a runtime restart/disconnect instead of vanishing with the VM. Adjust this
    # path if you want a different Drive folder.
    COLAB_OUTPUT_ROOT = "/content/drive/MyDrive/content_based_outputs/"
else:
    COLAB_DATA_DIR_CANDIDATES = ()
    COLAB_OUTPUT_ROOT = None  # local runs keep writing to the current working directory

print(f"IN_COLAB={IN_COLAB}")


IN_COLAB=False


In [4]:
from __future__ import annotations

import hashlib
import json
import random
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

CONFIG = {
    "data_dir_candidates": (
        "./processed_kindle/", "./proj2/processed_kindle/",
        *COLAB_DATA_DIR_CANDIDATES,  # from the Colab bootstrap cell above; empty off Colab
    ),
    "output_dir": "content_based_results_ALIGNED",

    # Shared content-text construction
    "description_truncate_chars": 1500,

    # Basic variant (TF-IDF hashing, no reduction)
    "basic_n_features": 2 ** 15,

    # Weighted-TFIDF (adopted from content_based_three_variants_weighted_tfidf_aligned.ipynb):
    # explicit per-field weights, searched on validation NDCG@10 in the field-weight
    # ablation further down. basic_field_weights is the all-1.0 control (numerically
    # identical to Basic-TFIDF); weighted_field_weight_candidates is the search grid.
    "basic_field_weights": {
        "title": 1.0,
        "categories": 1.0,
        "features": 1.0,
        "description": 1.0,
        "author": 0.0,
    },
    "weighted_field_weight_candidates": {
        # Full 14-point grid (restored from content_based_three_variants_weighted_tfidf_
        # aligned.ipynb; previously trimmed to 8 points for a quick runtime check).
        "title2_categories2": {"title": 2.0, "categories": 2.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title3_categories2": {"title": 3.0, "categories": 2.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_categories3": {"title": 2.0, "categories": 3.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_categories2_features1_5": {"title": 2.0, "categories": 2.0, "features": 1.5, "description": 1.0, "author": 0.0},
        "title2_categories2_5": {"title": 2.0, "categories": 2.5, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_categories3_5": {"title": 2.0, "categories": 3.5, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_categories4": {"title": 2.0, "categories": 4.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title1_5_categories3": {"title": 1.5, "categories": 3.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_5_categories3": {"title": 2.5, "categories": 3.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_25_categories2_75": {"title": 2.25, "categories": 2.75, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_25_categories3": {"title": 2.25, "categories": 3.0, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_5_categories2_75": {"title": 2.5, "categories": 2.75, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_5_categories3_25": {"title": 2.5, "categories": 3.25, "features": 1.0, "description": 1.0, "author": 0.0},
        "title2_75_categories3": {"title": 2.75, "categories": 3.0, "features": 1.0, "description": 1.0, "author": 0.0},
    },

    # Weighted-TFIDF text-representation upgrades (on top of the field-weight search):
    # bigrams capture short phrases ("psychological thriller") that unigram hashing
    # splits apart; sublinear_tf dampens very frequent terms. Basic-TFIDF is left
    # untouched (still unigram, linear tf) so it stays a clean, simple reference point --
    # only the already-winning Weighted-TFIDF path gets these on top.
    "weighted_tfidf_ngram_range": (1, 2),
    "weighted_tfidf_sublinear_tf": True,

    # Variant 1 (TF-IDF + SVD)
    "svd_max_features": 20_000,
    "svd_min_df": 3,
    "svd_n_components": 128,

    # Variant 1b (Field-Weighted TF-IDF + SVD): SVD dims to try on top of the
    # already field-weighted, bigram, sublinear-tf hashed counts (BEST_WEIGHTED_
    # FIELD_WEIGHTS from the field-weight search ablation below), to check whether
    # SVD helps once applied to the improved representation instead of plain TF-IDF.
    "weighted_svd_dim_candidates": [128, 256, 512],

    # Variant 2 (Sentence-BERT)
    "run_sbert": True,
    "sbert_model_name": "all-MiniLM-L6-v2",
    "sbert_batch_size": 32,

    # Shared recommendation parameters
    "n_neighbors_index": 50,
    "top_k": 10,
    "candidate_pool": 50,

    # User-profile weighting. Validation results in v7 did not show a material
    # benefit from upweighting verified purchases, so the aligned default is neutral.
    "verified_purchase_multiplier": 1.0,
    "rating_scale_max": 5.0,

    # Shared evaluation protocol. Keep evaluation_seed identical to the sequential
    # notebook so tied candidate scores are resolved identically across methods.
    "eval_k": 10,
    "primary_metric": "NDCG@10",
    "model_seed": 2026,
    "evaluation_seed": 2026,
    "diagnostic_n_users": 5000,
    # Raised from 50: with only 50 users x top_k=10, the theoretical max Catalog
    # Coverage is 500/81322=0.00615 -- the earlier 0.006 reading was ~97% of that
    # ceiling, so it reflected sample size, not real cross-user redundancy. 5000
    # users gives a ceiling of ~0.615, leaving enough room for the number to
    # actually distinguish "diverse" from "redundant" recommendations.
}

# Rooted under Drive on Colab (see COLAB_OUTPUT_ROOT in the bootstrap cell above) so
# every artifact this notebook writes survives a runtime restart; unchanged local-relative
# path otherwise.
OUTPUT_DIR = (
    Path(COLAB_OUTPUT_ROOT) / CONFIG["output_dir"]
    if IN_COLAB
    else Path(CONFIG["output_dir"])
)
DATA_DIR_CANDIDATES = tuple(Path(path) for path in CONFIG["data_dir_candidates"])
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.is_dir()), DATA_DIR_CANDIDATES[0])


def progress(message: str, done: bool = False) -> None:
    end = "\n" if done else "\r"
    print(message.ljust(100), end=end, flush=True)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


seed_everything(CONFIG["model_seed"])
print(
    f"data_dir={DATA_DIR.resolve()}\n"
    f"model_seed={CONFIG['model_seed']}, evaluation_seed={CONFIG['evaluation_seed']}, "
    f"primary_metric={CONFIG['primary_metric']}, run_sbert={CONFIG['run_sbert']}"
)


data_dir=/Users/lumi/PycharmProjects/PythonProject/comp9727/teamProject/processed_kindle
model_seed=2026, evaluation_seed=2026, primary_metric=NDCG@10, run_sbert=True


# Data Loading

Same shared loading logic used across all team methods: unified index mappings, merged item metadata (content + popularity stats), positive-only interaction file, and the official sampled evaluation files.

`require_files()` checks that every expected file exists before anything downstream tries to open it; `audit_protocol()` (next section) then checks that the *contents* line up.

In [5]:
REQUIRED_FILES = [
    "user2idx.json", "item2idx.json", "idx2user.json", "idx2item.json",
    "items_metadata_clean.csv", "item_metadata.csv", "interactions_clean.csv",
    "val.csv", "test.csv", "val_negatives.csv", "test_negatives.csv",
]


def require_files(data_dir: Path) -> None:
    missing = [name for name in REQUIRED_FILES if not (data_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing {len(missing)} required file(s) under {data_dir.resolve()}: {missing}"
        )


def normalise_split_name(split: str) -> str:
    aliases = {"val": "validation", "validation": "validation", "test": "test", "train": "train"}
    if split not in aliases:
        raise ValueError(f"Unknown split={split!r}; expected train, validation/val, or test")
    return aliases[split]


def load_mappings(data_dir: Path) -> dict:
    mappings = {}
    for name in ["user2idx", "item2idx", "idx2user", "idx2item"]:
        with open(data_dir / f"{name}.json", "r", encoding="utf-8") as f:
            mappings[name] = json.load(f)
    return mappings


ITEM_TEXT_FIELDS = ("title", "categories", "features", "description", "author")


def prepare_item_frame(items_df: pd.DataFrame, description_truncate_chars: int) -> pd.DataFrame:
    """Return a copy with every text field present, cleaned, and description truncated.
    Shared by build_unweighted_content_text() below and by FieldWeightedTFIDFVectorizer
    later, so both the unweighted baseline and the weighted variant clean text identically
    -- the only difference between them is the per-field weight, nothing else."""
    frame = items_df.copy()
    for column in ITEM_TEXT_FIELDS:
        if column not in frame.columns:
            frame[column] = ""
        frame[column] = frame[column].fillna("").astype(str)
    frame["categories"] = frame["categories"].str.replace(";", " ", regex=False)
    frame["description"] = frame["description"].str.slice(0, description_truncate_chars)
    return frame


def build_unweighted_content_text(
    items_df: pd.DataFrame,
    description_truncate_chars: int,
) -> pd.Series:
    """Each content field appears exactly once -- a genuinely unweighted baseline text.

    Adopted from content_based_three_variants_weighted_tfidf_aligned.ipynb: the previous
    version of this notebook built content_text as
    (title*2 + categories*2 + features + description), which meant Basic-TFIDF, SVD, and
    SBERT were all silently upweighting title/categories with no ablation ever having
    tested whether that specific 2x weighting was even a good choice (see the "对比口径说明"
    note below, now resolved). This function removes that hidden weighting; the properly
    isolated question "does upweighting title/categories help, and by how much" is instead
    answered by the dedicated Weighted-TFIDF variant and its field-weight search ablation
    further down, which can turn any field weight up or down independently of this baseline.
    """
    frame = prepare_item_frame(items_df, description_truncate_chars)
    return (
        frame["title"] + " "
        + frame["categories"] + " "
        + frame["features"] + " "
        + frame["description"]
    ).str.replace(r"\s+", " ", regex=True).str.strip()


def load_items(data_dir: Path, description_truncate_chars: int) -> pd.DataFrame:
    content_df = pd.read_csv(data_dir / "items_metadata_clean.csv")
    stats_df = pd.read_csv(data_dir / "item_metadata.csv")[
        ["item_idx", "author", "average_rating", "rating_number"]
    ]
    items_df = content_df.merge(stats_df, on="item_idx", how="left", validate="one_to_one")

    for col in ["title", "categories", "features", "description", "author"]:
        items_df[col] = items_df[col].fillna("")

    # Genuinely unweighted baseline text (previously title/categories were silently
    # duplicated *2 here -- see build_unweighted_content_text()'s docstring above for why
    # that changed). Basic-TFIDF, SVD, and SBERT below all read this same column, so all
    # three now start from a clean, unweighted representation.
    items_df["content_text"] = build_unweighted_content_text(items_df, description_truncate_chars)
    return items_df.sort_values("item_idx").reset_index(drop=True)


def load_interactions(data_dir: Path, split: str | None = None) -> pd.DataFrame:
    df = pd.read_csv(data_dir / "interactions_clean.csv")
    if split is not None:
        split = normalise_split_name(split)
        df = df[df["split"] == split].reset_index(drop=True)
    return df


def load_eval_positives(data_dir: Path, split: str) -> pd.DataFrame:
    split = normalise_split_name(split)
    fname = "val.csv" if split == "validation" else "test.csv"
    return pd.read_csv(data_dir / fname)


def load_eval_negatives(data_dir: Path, split: str) -> dict[int, list[int]]:
    split = normalise_split_name(split)
    fname = "val_negatives.csv" if split == "validation" else "test_negatives.csv"
    df = pd.read_csv(data_dir / fname)
    return {
        int(row.user_idx): [int(value) for value in row.neg_items.split()]
        for row in df.itertuples()
    }


# Protocol audit

Before fitting anything, check the assumptions the rest of the notebook relies on: index alignment between `items_df` and `item2idx`, matching user sets between the positive and negative evaluation files, exactly 100 distinct negatives per user with no overlap against the positive, and — the one that actually matters for validity — that no held-out val/test positive has leaked into `interactions_clean.csv`'s `train` split for that user. A failure here means the shared team files are inconsistent, not that this notebook's models are wrong; it is meant to fail loudly before any evaluation number is trusted.

In [6]:
def audit_protocol(
    mappings: dict,
    items_df: pd.DataFrame,
    train_interactions: pd.DataFrame,
    validation_interactions: pd.DataFrame,
    val_pos: pd.DataFrame,
    test_pos: pd.DataFrame,
    val_neg: dict,
    test_neg: dict,
) -> None:
    failures = []
    n_items = len(mappings["idx2item"])

    if items_df["item_idx"].tolist() != list(range(n_items)):
        failures.append(
            f"items_df item_idx is not the continuous range 0..{n_items - 1}; "
            f"actual rows={len(items_df)}"
        )

    train_sets = train_interactions.groupby("user_idx")["item_idx"].apply(set).to_dict()
    validation_map = dict(
        zip(
            validation_interactions["user_idx"].astype(int),
            validation_interactions["item_idx"].astype(int),
        )
    )
    val_target_map = dict(zip(val_pos["user_idx"].astype(int), val_pos["pos_idx"].astype(int)))
    test_target_map = dict(zip(test_pos["user_idx"].astype(int), test_pos["pos_idx"].astype(int)))

    if validation_map != val_target_map:
        failures.append("validation rows in interactions_clean.csv do not match val.csv")

    for name, pos_df, neg_dict in [
        ("validation", val_pos, val_neg),
        ("test", test_pos, test_neg),
    ]:
        pos_users = set(pos_df["user_idx"].astype(int))
        neg_users = set(neg_dict)
        if pos_users != neg_users:
            failures.append(
                f"{name}: positive and negative user sets differ "
                f"(missing={len(pos_users - neg_users)}, extra={len(neg_users - pos_users)})"
            )

        for row in pos_df.itertuples():
            user, positive = int(row.user_idx), int(row.pos_idx)
            negatives = neg_dict.get(user, [])
            if len(negatives) != 100 or len(set(negatives)) != 100:
                failures.append(
                    f"{name} user={user}: expected 100 distinct negatives; "
                    f"got {len(negatives)} ({len(set(negatives))} distinct)"
                )
            if positive in negatives:
                failures.append(f"{name} user={user}: positive item appears in negatives")
            if any(item < 0 or item >= n_items for item in [positive, *negatives]):
                failures.append(f"{name} user={user}: candidate item is outside 0..{n_items - 1}")

            # Fixed negatives must not include any known positive for this user.
            all_known_positives = set(train_sets.get(user, set()))
            all_known_positives.update({val_target_map[user], test_target_map[user]})
            if set(negatives) & all_known_positives:
                failures.append(f"{name} user={user}: negatives overlap the user's known positives")

    for user, val_item in val_target_map.items():
        if val_item in train_sets.get(user, set()):
            failures.append(f"validation user={user}: held-out item appears in training history")
    for user, test_item in test_target_map.items():
        if test_item in train_sets.get(user, set()) or test_item == val_target_map[user]:
            failures.append(f"test user={user}: held-out item leaks/repeats in earlier history")

    if failures:
        raise AssertionError(
            f"Protocol audit failed with {len(failures)} issue(s); first five:\n"
            + "\n".join(failures[:5])
        )

    print(
        f"Protocol audit passed: {n_items:,} items, "
        f"validation={len(val_pos):,} users, test={len(test_pos):,} users, "
        f"train_interactions={len(train_interactions):,}. "
        "Validation context=train; test context=train+validation."
    )


In [7]:
require_files(DATA_DIR)

mappings = load_mappings(DATA_DIR)
items_df = load_items(DATA_DIR, CONFIG["description_truncate_chars"])
all_interactions = load_interactions(DATA_DIR)
train_interactions = all_interactions[all_interactions["split"] == "train"].reset_index(drop=True)
validation_interactions = all_interactions[
    all_interactions["split"] == "validation"
].reset_index(drop=True)

# Chronological evaluation contexts:
#   validation profile = train
#   test profile       = train + validation interaction
test_profile_interactions = pd.concat(
    [train_interactions, validation_interactions],
    ignore_index=True,
)

n_users = len(mappings["idx2user"])
n_items = len(mappings["idx2item"])

val_positives = load_eval_positives(DATA_DIR, "validation")
test_positives = load_eval_positives(DATA_DIR, "test")
val_negatives = load_eval_negatives(DATA_DIR, "validation")
test_negatives = load_eval_negatives(DATA_DIR, "test")

audit_protocol(
    mappings,
    items_df,
    train_interactions,
    validation_interactions,
    val_positives,
    test_positives,
    val_negatives,
    test_negatives,
)

# Shared popularity is learned only from training interactions, exactly as in the
# sequential notebook. Metadata rating_number is retained for display only.
train_item_counts = (
    train_interactions.groupby("item_idx")
    .size()
    .reindex(range(n_items), fill_value=0)
    .astype(np.int64)
    .to_numpy()
)
items_df["train_interaction_count"] = train_item_counts
items_df["popularity_score"] = np.log1p(train_item_counts)

train_interacted_items = (
    train_interactions.groupby("user_idx")["item_idx"].apply(set).to_dict()
)
test_interacted_items = (
    test_profile_interactions.groupby("user_idx")["item_idx"].apply(set).to_dict()
)

description_missing_rate = (items_df["description"].str.len() == 0).mean()
print(
    f"items={n_items:,}, users={n_users:,}, train={len(train_interactions):,}, "
    f"validation_context_additions={len(validation_interactions):,}, "
    f"description_missing={description_missing_rate:.1%}"
)


Protocol audit passed: 81,322 items, validation=64,867 users, test=64,867 users, train_interactions=884,136. Validation context=train; test context=train+validation.
items=81,322, users=64,867, train=884,136, validation_context_additions=64,867, description_missing=39.8%


# Baselines

`Popularity` ranks every user identically by the team's shared `popularity_score` (`average_rating * log1p(rating_number)`) and uses no content features or interaction history at all. It exists to answer one question before the content-based variants below are trusted: **does using book content actually help, or would a non-personalised top-seller list already win?** If `Basic-TFIDF` cannot beat this, the content signal is not adding value.

> 这里跑出来的 Popularity 只是本notebook内部用于对照的最简单实现，团队里负责 Most Popular 的同学大概率已经用同一套 `val.csv`/`test.csv`/`val_negatives.csv`/`test_negatives.csv` 跑出了她自己的版本——如果口径一致，两边数字应该基本一致，可以互相验证；如果你们最终报告要用统一的 Popularity 数字，直接拿她那份替换这里的结果即可，不需要两份都保留。

In [8]:
class PopularityRecommender:
    '''Common non-personalised baseline based only on training interaction counts.'''

    def __init__(self, items_df: pd.DataFrame, interacted_items: dict[int, set[int]]):
        self.items_df = items_df.set_index("item_idx")
        self.interacted_items = interacted_items
        self.ranked_items = (
            items_df.sort_values(
                ["popularity_score", "item_idx"],
                ascending=[False, True],
            )
            ["item_idx"]
            .astype(int)
            .tolist()
        )

    def recommend(self, user_idx: int, k: int | None = None, candidate_pool: int | None = None):
        k = k or CONFIG["top_k"]
        seen = self.interacted_items.get(user_idx, set())
        item_ids = [item for item in self.ranked_items if item not in seen][:k]
        return [
            {
                "item_idx": item,
                "reason": "popular",
                "similarity": 0.0,
                "title": self.items_df.loc[item, "title"],
            }
            for item in item_ids
        ]

    def score_candidates(self, user_idx: int, candidate_item_idxs: list[int]) -> np.ndarray:
        # Candidate-set evaluation negatives already exclude every known positive.
        return self.items_df.loc[candidate_item_idxs, "popularity_score"].to_numpy(dtype=float)


# Variant Implementations (content-based, rungs 1-3)

All three vectorizers expose the same interface — `fit_transform(items_df)`, `build_index(n_neighbors)`, `query_topk(query_vectors, k)` — so they plug into the same recommender/evaluation code below without any downstream changes.

### Basic: TF-IDF (hashing trick) + Cosine Similarity

Uses `HashingVectorizer` rather than `TfidfVectorizer`: building a full vocabulary dictionary for 81k items (some descriptions run past 250k characters before truncation) spikes memory past what this machine (3.9GB RAM) can hold. Hashing tokens into a fixed number of buckets avoids ever materializing that dictionary, at the cost of losing interpretable feature names.

In [9]:
class TFIDFHashingVectorizer:
    def __init__(self, n_features: int):
        self.hv = HashingVectorizer(n_features=n_features, stop_words="english",
                                     alternate_sign=False, norm=None)
        self.tt = TfidfTransformer()
        self.item_matrix = None
        self.nn_index = None
        self.item_idx_order = None

    def fit_transform(self, items_df: pd.DataFrame):
        items_df = items_df.sort_values("item_idx").reset_index(drop=True)
        self.item_idx_order = items_df["item_idx"].to_numpy()
        counts = self.hv.transform(items_df["content_text"])
        self.item_matrix = self.tt.fit_transform(counts)
        return self.item_matrix

    def build_index(self, n_neighbors: int):
        self.nn_index = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
        self.nn_index.fit(self.item_matrix)
        return self.nn_index

    def query_topk(self, query_vectors, k: int):
        return self.nn_index.kneighbors(query_vectors, n_neighbors=k)

    def transform_new_items(self, content_texts: list[str]):
        """Embed brand-new item text with no refit at all: HashingVectorizer has no
        vocabulary to learn, and TfidfTransformer's already-fitted idf weights are reused
        as-is. This is the item-cold-start path -- a book with zero interactions can be
        scored the moment its title/description exist, no retraining required."""
        counts = self.hv.transform(content_texts)
        return self.tt.transform(counts)


### Weighted-TFIDF: explicit per-field weights (isolates the effect of upweighting title/categories)

Adopted from `content_based_three_variants_weighted_tfidf_aligned.ipynb`. Same hashing
dimension, stop-word policy, `TfidfTransformer`, and cosine metric as Basic-TFIDF above --
the only thing that differs is an explicit numerical weight per metadata field
(`title`, `categories`, `features`, `description`, `author`), applied to that field's hashed
term counts *before* they are summed and handed to `TfidfTransformer`. With `title=1.0,
categories=1.0, features=1.0, description=1.0, author=0.0` this is numerically identical to
Basic-TFIDF; the field-weight search ablation further down instead tries weight combinations
selected on validation `NDCG@10`, giving a real, single-variable answer to the question the
old implicit `title*2 + categories*2` baseline never actually tested.


**Update:** now also uses `ngram_range=(1, 2)` (hashes adjacent word pairs, e.g. "psychological thriller", that unigram-only hashing always splits apart) and `sublinear_tf=True` (dampens very frequent terms) -- both configurable via `CONFIG["weighted_tfidf_ngram_range"]`/`CONFIG["weighted_tfidf_sublinear_tf"]`, default on for every Weighted-TFIDF run below. Basic-TFIDF is deliberately left untouched (still unigram, linear tf) so it stays a clean, simple reference point.


In [10]:
class FieldWeightedTFIDFVectorizer:
    """Hashed TF-IDF with explicit numerical weights for individual metadata fields.

    A weight multiplies that field's hashed term counts before TfidfTransformer is fitted
    -- with the default linear term frequency this is equivalent to upweighting that field,
    while keeping memory bounded because (like TFIDFHashingVectorizer above) no vocabulary
    dictionary is ever materialised. Operates on the raw per-field columns directly
    (prepare_item_frame() above), not on the combined content_text string, so this variant
    never depends on how content_text happens to be built.
    """

    def __init__(
        self,
        n_features: int,
        field_weights: dict[str, float],
        description_truncate_chars: int,
        ngram_range: tuple[int, int] = (1, 1),
        sublinear_tf: bool = False,
    ):
        unknown = set(field_weights) - set(ITEM_TEXT_FIELDS)
        if unknown:
            raise ValueError(f"Unknown item text fields: {sorted(unknown)}")
        if not any(float(weight) > 0 for weight in field_weights.values()):
            raise ValueError("At least one field weight must be positive")
        if any(float(weight) < 0 for weight in field_weights.values()):
            raise ValueError("Field weights cannot be negative")

        self.field_weights = {
            field: float(field_weights.get(field, 0.0))
            for field in ITEM_TEXT_FIELDS
        }
        self.description_truncate_chars = int(description_truncate_chars)
        self.ngram_range = tuple(ngram_range)
        self.sublinear_tf = bool(sublinear_tf)
        # ngram_range=(1, 2) also hashes adjacent word pairs ("psychological thriller"),
        # which unigram-only hashing above always splits into two independent tokens.
        # Defaults stay (1, 1)/False so this class is still usable as a drop-in
        # unweighted-and-unigram baseline if ever needed; CONFIG turns both on for the
        # actual Weighted-TFIDF runs (see the field-weight search ablation).
        self.hv = HashingVectorizer(
            n_features=n_features,
            stop_words="english",
            alternate_sign=False,
            norm=None,
            ngram_range=self.ngram_range,
        )
        self.tt = TfidfTransformer(norm="l2", sublinear_tf=self.sublinear_tf)
        self.item_matrix = None
        self.nn_index = None
        self.item_idx_order = None

    def _weighted_counts(self, items_df: pd.DataFrame):
        frame = prepare_item_frame(items_df, self.description_truncate_chars)
        weighted_counts = None
        for field, weight in self.field_weights.items():
            if weight <= 0:
                continue
            field_counts = self.hv.transform(frame[field])
            if weight != 1.0:
                field_counts = field_counts.multiply(weight)
            weighted_counts = (
                field_counts if weighted_counts is None else weighted_counts + field_counts
            )
        if weighted_counts is None:
            raise RuntimeError("No positive field weight was available")
        return weighted_counts.tocsr()

    def fit_transform(self, items_df: pd.DataFrame):
        ordered = items_df.sort_values("item_idx").reset_index(drop=True)
        self.item_idx_order = ordered["item_idx"].to_numpy()
        counts = self._weighted_counts(ordered)
        self.item_matrix = self.tt.fit_transform(counts).astype(np.float32)
        return self.item_matrix

    def build_index(self, n_neighbors: int):
        n_neighbors = min(int(n_neighbors), int(self.item_matrix.shape[0]))
        self.nn_index = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
        self.nn_index.fit(self.item_matrix)
        return self.nn_index

    def query_topk(self, query_vectors, k: int):
        k = min(int(k), int(self.item_matrix.shape[0]))
        return self.nn_index.kneighbors(query_vectors, n_neighbors=k)

    def transform_new_items(self, new_items: pd.DataFrame):
        """Item-cold-start path: reuse the fitted IDF values without any refit."""
        counts = self._weighted_counts(new_items)
        return self.tt.transform(counts).astype(np.float32)


### Variant 1: TF-IDF + TruncatedSVD (128-dim) + Cosine Similarity

Dense 128-dim item vectors keep downstream user-profile construction to a small dense matmul instead of a sparse x sparse multiply — important at this catalog size on memory-constrained machines.

In [11]:
class SVDContentVectorizer:
    """在原有实现基础上，额外保留降维前的原始TF-IDF向量和词表，专门用来生成
    可读的关键词解释；打分排序仍然只用降维后的 self.item_matrix，不受影响。"""

    def __init__(self, max_features: int, min_df: int, n_components: int):
        self.tfidf = TfidfVectorizer(max_features=max_features, min_df=min_df, stop_words="english")
        self.svd = TruncatedSVD(n_components=n_components, random_state=CONFIG["model_seed"])
        self.item_matrix = None
        self.nn_index = None
        self.item_idx_order = None
        self.raw_tfidf_matrix = None      # 新增：降维前的原始TF-IDF，仅用于生成解释
        self.feature_names = None          # 新增：词表，用于把向量维度映射回具体的词

    def fit_transform(self, items_df: pd.DataFrame) -> np.ndarray:
        items_df = items_df.sort_values("item_idx").reset_index(drop=True)
        self.item_idx_order = items_df["item_idx"].to_numpy()
        tfidf_matrix = self.tfidf.fit_transform(items_df["content_text"])
        self.raw_tfidf_matrix = tfidf_matrix
        self.feature_names = self.tfidf.get_feature_names_out()
        reduced = self.svd.fit_transform(tfidf_matrix).astype(np.float32)
        self.item_matrix = normalize(reduced)
        return self.item_matrix

    def build_index(self, n_neighbors: int):
        self.nn_index = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
        self.nn_index.fit(self.item_matrix)
        return self.nn_index

    def query_topk(self, query_vectors: np.ndarray, k: int):
        return self.nn_index.kneighbors(query_vectors, n_neighbors=k)

    def transform_new_items(self, content_texts: list[str]):
        """Reuse the already-fitted vocabulary + SVD components (transform only, never
        fit) -- so a new item never needs the NearestNeighbors index rebuilt to be scored."""
        tfidf_vec = self.tfidf.transform(content_texts)
        reduced = self.svd.transform(tfidf_vec).astype(np.float32)
        return normalize(reduced)


### Variant 1b: Field-Weighted TF-IDF + SVD

`TFIDF-SVD-TUNED` above reduces a **plain, unweighted** TF-IDF matrix. `Weighted-TFIDF-TUNED`
(the current best model) never gets dimensionality reduction. Neither ablation has ever
tested the combination -- this variant reuses `FieldWeightedTFIDFVectorizer`'s weighted,
n-gram, sublinear-tf hashed counts as SVD's input, so we can check whether SVD's denoising
still helps once the input signal is already improved, or whether -- as with the plain-TFIDF
case -- reducing dimensions loses more than it gains here.

In [12]:
class FieldWeightedSVDContentVectorizer:
    """SVD dimensionality reduction stacked on top of field-weighted hashed TF-IDF.

    Reuses FieldWeightedTFIDFVectorizer internally to build the weighted, L2-normalised
    sparse matrix (same field weights, ngram_range, sublinear_tf as Weighted-TFIDF-TUNED),
    then applies TruncatedSVD to it -- TruncatedSVD works directly on sparse input, so no
    densification step is needed. This isolates a single question: does SVD help or hurt
    once it is applied to the *already field-weighted* representation, rather than to the
    plain unweighted TF-IDF that TFIDF-SVD-TUNED reduces?
    """

    def __init__(
        self,
        n_features: int,
        field_weights: dict[str, float],
        description_truncate_chars: int,
        n_components: int,
        ngram_range: tuple[int, int] = (1, 1),
        sublinear_tf: bool = False,
    ):
        self.weighted_tfidf = FieldWeightedTFIDFVectorizer(
            n_features,
            field_weights,
            description_truncate_chars,
            ngram_range=ngram_range,
            sublinear_tf=sublinear_tf,
        )
        self.svd = TruncatedSVD(n_components=n_components, random_state=CONFIG["model_seed"])
        self.item_matrix = None
        self.nn_index = None
        self.item_idx_order = None

    def fit_transform(self, items_df: pd.DataFrame) -> np.ndarray:
        weighted_matrix = self.weighted_tfidf.fit_transform(items_df)  # sparse, already L2-normalised
        self.item_idx_order = self.weighted_tfidf.item_idx_order
        reduced = self.svd.fit_transform(weighted_matrix).astype(np.float32)
        self.item_matrix = normalize(reduced)
        return self.item_matrix

    def build_index(self, n_neighbors: int):
        n_neighbors = min(int(n_neighbors), int(self.item_matrix.shape[0]))
        self.nn_index = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
        self.nn_index.fit(self.item_matrix)
        return self.nn_index

    def query_topk(self, query_vectors: np.ndarray, k: int):
        k = min(int(k), int(self.item_matrix.shape[0]))
        return self.nn_index.kneighbors(query_vectors, n_neighbors=k)

    def transform_new_items(self, new_items: pd.DataFrame):
        """Item-cold-start path: reuse fitted IDF values and SVD components, never refit."""
        weighted_counts = self.weighted_tfidf.transform_new_items(new_items)
        reduced = self.svd.transform(weighted_counts).astype(np.float32)
        return normalize(reduced)


> **⚠️ 对比口径说明（Basic vs Variant 1）**
>
> `Basic` 用 `HashingVectorizer`（32768个hash桶），`Variant 1` 用 `TfidfVectorizer`（20000词表，`min_df=3`）再做SVD降维 —— **两者的基础特征提取方式本身就不同**，不是"同一份TF-IDF向量，一个加SVD一个不加"这种严格控制单一变量的对照实验。
>
> 差异来源除了"是否降维"之外，还包括：
> - 特征构造方式（哈希 vs 显式词表）
> - 特征维度（32768 vs 20000）
> - 是否存在哈希碰撞（Basic有，SVD路径没有）
> - `min_df=3` 只在SVD路径生效
>
> 之所以Basic不用标准`TfidfVectorizer`，是内存工程约束——完整词表在81322个item、部分description超长（最长25万字符）的情况下会导致内存溢出（我们实测过）。
>
> **报告里请勿写"Variant 1只是在Basic基础上加了SVD"**，更准确的表述是：
> *"The basic method used hashed TF-IDF for memory efficiency, while the SVD variant used a vocabulary-based TF-IDF representation before dimensionality reduction. The two therefore differ in feature construction as well as in the presence of SVD."*
>
> **Update: this is now resolved.** `load_items()` above was changed to build `content_text`
> via `build_unweighted_content_text()` -- title and categories no longer get a hidden `*2`
> duplication, so Basic-TFIDF (and SVD and SBERT, which read the same column) now start from
> a genuinely unweighted representation. The properly isolated question this caveat used to
> block -- "does upweighting title/categories actually help, and by how much" -- is answered
> instead by the dedicated **Weighted-TFIDF** variant (`FieldWeightedTFIDFVectorizer` below)
> and its field-weight search ablation further down, which varies only the field weights
> against the exact same hashing dimension, stop-word policy, and TF-IDF transformer as
> Basic-TFIDF -- a real single-variable comparison this time.
>
> Anything computed *before* this change (including the Basic-TFIDF/SVD/SBERT numbers
> discussed earlier in this notebook's history) used the old implicit-weighted text and is
> **not directly comparable** to numbers computed after it. Re-running this notebook top to
> bottom makes everything internally consistent again under the new unweighted baseline; the
> qualitative conclusions from the earlier ablations (dimension 256 beating smaller SVDs,
> including author helping slightly, `verified_purchase_multiplier=1.0` being best) are
> unlikely to flip, but the exact numbers should be re-verified rather than quoted from before
> this change.


### Variant 2: Sentence-BERT (all-MiniLM-L6-v2) + Cosine Similarity

**Requires internet access** to download the pretrained model on first run. Guarded by `CONFIG["run_sbert"]` — currently defaults to `True`, so this variant runs as part of every full pass; set it to `False` if you want the rest of the notebook to execute without internet access or the sentence-transformers dependency. The setup cell immediately below auto-installs `sentence-transformers` (only when `run_sbert` is `True` and the import fails), so this runs unattended on Colab or any fresh environment.

**磁盘缓存**：`SBERTContentVectorizer` 会把编码好的item向量缓存到 `outputs/sbert_cache/`（按模型名+content_text的SHA1指纹命名文件）。只要 `content_text` 没变，重跑notebook时会直接读缓存，不用重新等 `all-mpnet-base-v2` 那~20分钟的encode。如果之后又改了 `content_text`（比如换字段权重），指纹会自动变化，缓存自动失效重新编码，不会用旧向量算出错的分数。

Expected to help most on the ~60% of items that have a real `description` (semantic similarity beyond keyword overlap); on title/categories-only items the advantage over TF-IDF is likely smaller, since there is less text for a sentence embedding to exploit.

In [13]:
if CONFIG["run_sbert"]:
    try:
        import sentence_transformers  # noqa: F401
    except ImportError:
        import subprocess
        import sys

        progress("sentence-transformers not found -- installing (requires internet)...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"],
            check=True,
        )
        import sentence_transformers  # noqa: F401
    print("sentence-transformers available, version:", sentence_transformers.__version__)

    import torch
    if torch.cuda.is_available():
        print(f"GPU detected: {torch.cuda.get_device_name(0)} -- SBERT encoding will use it.")
    else:
        print(
            "No GPU detected (torch.cuda.is_available()=False) -- SBERT encoding will run "
            "on CPU and be very slow (hours for mpnet on ~81k items). On Colab: Runtime > "
            "Change runtime type > GPU (T4 is fine) > Save, then Runtime > Disconnect and "
            "delete runtime, reconnect, and rerun this cell -- just changing the setting "
            "without reconnecting a fresh runtime keeps the old CPU-only session running."
        )
else:
    print("CONFIG['run_sbert'] is False -- skipping the sentence-transformers install/check.")


/opt/anaconda3/envs/comp9727/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence-transformers available, version: 5.6.1
No GPU detected (torch.cuda.is_available()=False) -- SBERT encoding will run on CPU and be very slow (hours for mpnet on ~81k items). On Colab: Runtime > Change runtime type > GPU (T4 is fine) > Save, then Runtime > Disconnect and delete runtime, reconnect, and rerun this cell -- just changing the setting without reconnecting a fresh runtime keeps the old CPU-only session running.


In [14]:
class SBERTContentVectorizer:
    def __init__(self, model_name: str, batch_size: int = 64, cache_dir: Path | str | None = None):
        from sentence_transformers import SentenceTransformer  # imported lazily: optional dependency
        import torch

        # Explicit device selection instead of relying on SentenceTransformer's own
        # auto-detection -- printed so it's immediately visible in the cell output
        # whether this run is actually on GPU, instead of having to infer it later from
        # how slow/fast the encode() progress bar turns out to be.
        device = "cuda" if torch.cuda.is_available() else "cpu"
        if device == "cpu":
            progress(
                "[SBERT] WARNING: no GPU detected (torch.cuda.is_available()=False) -- "
                "encoding all ~81k items on CPU will be very slow (hours for mpnet). "
                "On Colab: Runtime > Change runtime type > GPU, then reconnect and rerun.",
                done=True,
            )
        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        progress(f"[SBERT-{model_name}] using device={device}", done=True)
        self.batch_size = batch_size
        self.item_matrix = None
        self.nn_index = None
        self.item_idx_order = None
        # Encoding all ~81k items takes ~2 minutes for MiniLM and ~20 minutes for mpnet --
        # cache to disk so re-running this notebook doesn't re-pay that cost every time.
        self.cache_dir = Path(cache_dir) if cache_dir is not None else OUTPUT_DIR / "sbert_cache"

    @staticmethod
    def _content_fingerprint(content_texts: pd.Series) -> str:
        """Hashes the *actual text being encoded*, not just row count, so a cache left over
        from before a content_text change (e.g. the earlier fix that made content_text
        genuinely unweighted) is detected as stale and never silently reused -- a wrong
        cache hit here would be a silent correctness bug, not just a slow one."""
        hasher = hashlib.sha1()
        for text in content_texts:
            hasher.update(text.encode("utf-8"))
            hasher.update(b"\x00")
        return hasher.hexdigest()[:16]

    def _cache_paths(self, fingerprint: str) -> tuple[Path, Path]:
        safe_model_name = self.model_name.replace("/", "_")
        base = self.cache_dir / f"{safe_model_name}_{fingerprint}"
        return base.with_suffix(".embeddings.npy"), base.with_suffix(".item_idx.npy")

    def fit_transform(self, items_df: pd.DataFrame) -> np.ndarray:
        ordered = items_df.sort_values("item_idx").reset_index(drop=True)
        self.item_idx_order = ordered["item_idx"].to_numpy()

        fingerprint = self._content_fingerprint(ordered["content_text"])
        embeddings_path, item_idx_path = self._cache_paths(fingerprint)

        if embeddings_path.exists() and item_idx_path.exists():
            cached_embeddings = np.load(embeddings_path)
            cached_item_idx = np.load(item_idx_path)
            if (
                cached_embeddings.shape[0] == len(ordered)
                and np.array_equal(cached_item_idx, self.item_idx_order)
            ):
                progress(
                    f"[SBERT-{self.model_name}] loaded cached embeddings from "
                    f"{embeddings_path.name} (delete this file to force re-encoding)",
                    done=True,
                )
                self.item_matrix = cached_embeddings.astype(np.float32)
                return self.item_matrix
            # Fingerprint matched but shape/order didn't (should not happen in practice) --
            # fail safe by falling through to a fresh encode rather than trusting the cache.

        embeddings = self.model.encode(
            ordered["content_text"].tolist(),
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,  # L2-normalize so dot product == cosine similarity
        )
        self.item_matrix = embeddings.astype(np.float32)

        self.cache_dir.mkdir(parents=True, exist_ok=True)
        np.save(embeddings_path, self.item_matrix)
        np.save(item_idx_path, self.item_idx_order)
        progress(f"[SBERT-{self.model_name}] cached embeddings to {embeddings_path.name}", done=True)

        return self.item_matrix

    def build_index(self, n_neighbors: int):
        self.nn_index = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
        self.nn_index.fit(self.item_matrix)
        return self.nn_index

    def query_topk(self, query_vectors: np.ndarray, k: int):
        return self.nn_index.kneighbors(query_vectors, n_neighbors=k)

    def transform_new_items(self, content_texts: list[str]):
        """Frozen pretrained weights: a new item is embedded exactly like any training
        item, so there is no notion of item cold-start for this vectorizer at all."""
        embeddings = self.model.encode(
            content_texts, batch_size=self.batch_size,
            convert_to_numpy=True, normalize_embeddings=True,
        )
        return embeddings.astype(np.float32)


> **⚠️ SBERT 的双重截断问题**
>
> `CONFIG["description_truncate_chars"]=1500` 是我们为TF-IDF/SVD路径设计的截断（防止极端长description拖垮内存和引入噪声）。但 `all-MiniLM-L6-v2` / `all-mpnet-base-v2` 这类SBERT模型**自身也有max_seq_length限制**（前者约256 token≈200个英文单词，后者也是同一量级），远小于1500字符能容纳的文本量。
>
> 也就是说：**我们截到1500字符 ≠ SBERT模型实际处理了这1500字符**，模型内部会再做一次更激进的截断，真正"看到"的文本比我们以为的要短得多。
>
> 这是SBERT在本项目中表现不如TF-IDF/SVD的一个可能原因（之前分析里遗漏的一点），值得写进Reflection部分："SBERT模型自身的token长度限制可能进一步压缩了它实际能利用的内容信息，这是TF-IDF/SVD不存在的额外损失。"

# User Profiles

`weight(interaction) = (rating / 5.0) * (1.2 if verified_purchase else 1.0)`, shared across all three variants for a fair comparison. Two implementations are provided:

- `LazyUserProfiles` — computes each user's profile on demand from their handful of train interactions. Used for the **Basic** variant, where item vectors are large sparse vectors and precomputing all 64,867 user profiles at once is memory-heavy (summed rows are much denser than any single item's vector).
- `build_user_profiles_dense` — a single vectorized sparse-dense matmul, cheap because the item vectors are already small dense vectors (SVD / SBERT). Used for **Variant 1** and **Variant 2**.

In [15]:
class LazyUserProfiles:
    def __init__(
        self,
        interactions_df,
        item_matrix,
        item_idx_to_row,
        verified_purchase_multiplier: float,
    ):
        df = interactions_df.copy()
        df["row"] = df["item_idx"].map(item_idx_to_row)
        df = df.dropna(subset=["row"])
        df["row"] = df["row"].astype(int)

        rating_weight = df["rating"].to_numpy() / CONFIG["rating_scale_max"]
        verified_mult = np.where(
            df["verified_purchase"].to_numpy(dtype=bool),
            verified_purchase_multiplier,
            1.0,
        )
        df["weight"] = rating_weight * verified_mult

        self.user_rows = df.groupby("user_idx")["row"].apply(list).to_dict()
        self.user_weights = df.groupby("user_idx")["weight"].apply(list).to_dict()
        self.interacted_items = df.groupby("user_idx")["item_idx"].apply(set).to_dict()
        self.item_matrix = item_matrix
        self.vocab_size = item_matrix.shape[1]
        self.is_sparse = sp.issparse(item_matrix)

    def __getitem__(self, user_idx: int):
        rows = self.user_rows.get(user_idx)
        if not rows:
            if self.is_sparse:
                return sp.csr_matrix((1, self.vocab_size))
            return np.zeros((1, self.vocab_size), dtype=np.float32)

        weights = np.asarray(self.user_weights[user_idx])
        vecs = self.item_matrix[rows]
        if self.is_sparse:
            weighted = sp.diags(weights) @ vecs
            profile = sp.csr_matrix(weighted.sum(axis=0))
            norm = np.sqrt(profile.multiply(profile).sum())
            if norm > 0:
                profile = profile / norm
            return profile

        profile = (vecs * weights[:, None]).sum(axis=0, keepdims=True)
        norm = np.linalg.norm(profile)
        if norm > 0:
            profile = profile / norm
        return profile.astype(np.float32)


def build_user_profiles_dense(
    interactions_df,
    item_matrix,
    n_users,
    item_idx_to_row,
    verified_purchase_multiplier: float,
):
    df = interactions_df.copy()
    df["row"] = df["item_idx"].map(item_idx_to_row)
    df = df.dropna(subset=["row"])
    df["row"] = df["row"].astype(int)

    rating_weight = df["rating"].to_numpy() / CONFIG["rating_scale_max"]
    verified_mult = np.where(
        df["verified_purchase"].to_numpy(dtype=bool),
        verified_purchase_multiplier,
        1.0,
    )
    weight = rating_weight * verified_mult

    n_matrix_items = item_matrix.shape[0]
    interaction_weights = sp.coo_matrix(
        (weight, (df["user_idx"].to_numpy(), df["row"].to_numpy())),
        shape=(n_users, n_matrix_items),
    ).tocsr()
    profiles = interaction_weights @ item_matrix

    norms = np.linalg.norm(profiles, axis=1)
    norms[norms == 0] = 1.0
    profiles = (profiles.T / norms).T.astype(np.float32)
    interacted_items = df.groupby("user_idx")["item_idx"].apply(set).to_dict()

    class DenseProfileStore:
        is_sparse = False

        def __init__(self, matrix):
            self._matrix = matrix

        def __getitem__(self, index):
            return self._matrix[index].reshape(1, -1)

    return DenseProfileStore(profiles), interacted_items


# Recommendation Generation

Shared across all variants — Top-K recommendations with a `content-similar` reason, falling back to popularity ranking (`popular` reason) for cold-start users.

**Update:** `recommend()` now applies MMR (Maximal Marginal Relevance) reranking by default (`diversify=True`) over the `candidate_pool`-sized similarity-ranked pool, trading a small amount of raw similarity for less redundant top-10 lists (reusing `item_metadata_similarity()` from the Evaluation section below as the redundancy penalty -- same formula the sequential notebook's own `mmr_rerank()` uses). This only affects what `recommend()` shows a user; `score_candidates()` -- and therefore every Hit@10/NDCG@10/MRR number in this notebook -- is completely unaffected. Pass `diversify=False` to get the old pure-similarity ordering back.


In [16]:
class ContentRecommender:
    def __init__(self, vectorizer, items_df, profiles, interacted_items, item_idx_to_row):
        self.vectorizer = vectorizer
        self.items_df = items_df.set_index("item_idx")
        self.profiles = profiles
        self.interacted_items = interacted_items
        self.item_idx_to_row = item_idx_to_row
        self.row_to_item_idx = vectorizer.item_idx_order
        self.is_sparse = getattr(profiles, "is_sparse", None)
        if self.is_sparse is None:
            self.is_sparse = sp.issparse(profiles)
        self.popularity_rank = items_df.sort_values("popularity_score", ascending=False)["item_idx"].tolist()
        self.cold_start_builder = None  # attach via attach_cold_start() for new-user handling
        self.pending_new_items = {}  # item_idx -> vector, for items registered via register_new_item()

    def _get_profile_vec(self, user_idx: int):
        vec = self.profiles[user_idx]
        if self.is_sparse:
            return vec, vec.nnz == 0
        vec = vec.reshape(1, -1)
        return vec, not np.any(vec)

    def _popularity_fallback(self, exclude: set, k: int):
        recs = []
        for item_idx in self.popularity_rank:
            if item_idx in exclude:
                continue
            recs.append(item_idx)
            if len(recs) == k:
                break
        return recs

    def _mmr_select(
        self,
        candidates: list[tuple[int, float]],
        k: int,
        relevance_weight: float = 0.85,
    ) -> list[tuple[int, float]]:
        """Greedy Maximal Marginal Relevance rerank -- same formula as the sequential
        notebook's mmr_rerank(), reusing item_metadata_similarity() (category/author
        overlap, defined in the Evaluation section below) as the redundancy penalty.
        Only reorders what recommend() shows a user; score_candidates() -- and therefore
        every offline Hit@10/NDCG@10/MRR number in this notebook -- is untouched."""
        if len(candidates) <= k:
            return candidates
        scores = np.array([s for _, s in candidates], dtype=np.float64)
        score_range = max(scores.max() - scores.min(), 1e-12)
        relevance = (scores - scores.min()) / score_range

        selected: list[int] = []
        remaining = list(range(len(candidates)))
        while len(selected) < k and remaining:
            def mmr_value(position):
                redundancy = max(
                    (
                        item_metadata_similarity(self.items_df, candidates[position][0], candidates[chosen][0])
                        for chosen in selected
                    ),
                    default=0.0,
                )
                return relevance_weight * relevance[position] - (1 - relevance_weight) * redundancy

            best = max(remaining, key=mmr_value)
            selected.append(best)
            remaining.remove(best)
        return [candidates[i] for i in selected]

    def recommend(
        self,
        user_idx: int,
        k: int = None,
        candidate_pool: int = None,
        diversify: bool = True,
        mmr_relevance_weight: float = 0.85,
    ):
        k = k or CONFIG["top_k"]
        candidate_pool = candidate_pool or CONFIG["candidate_pool"]
        seen = self.interacted_items.get(user_idx, set())
        profile_vec, is_cold = self._get_profile_vec(user_idx)

        if is_cold:
            item_ids = self._popularity_fallback(seen, k)
            return [{"item_idx": i, "reason": "popular", "similarity": 0.0,
                      "title": self.items_df.loc[i, "title"]} for i in item_ids]

        dist, idx = self.vectorizer.query_topk(profile_vec, k=candidate_pool)
        dist, idx = dist[0], idx[0]
        sim = 1 - dist

        candidates = []
        for row, s in zip(idx, sim):
            item_idx = int(self.row_to_item_idx[row])
            if item_idx in seen:
                continue
            candidates.append((item_idx, float(s)))

        # Merge in any not-yet-indexed new items (see register_new_item()). The fitted
        # NearestNeighbors index is a static snapshot from build_index() -- rebuilding it
        # for every new arrival would be wasteful, so pending items are scored directly
        # against this user's profile and merged with the ANN index's candidates here.
        # Without this, score_new_item() only ever produced an isolated similarity number
        # that could never actually surface in a real recommend() call.
        for item_idx, new_vector in self.pending_new_items.items():
            if item_idx in seen:
                continue
            candidates.append((item_idx, self._new_item_similarity(new_vector, profile_vec)))

        # MMR reranking trades a small amount of pure similarity for less redundant
        # top-10 lists -- see the Cold-Start/catalogue-diagnostics numbers earlier in this
        # notebook (Intra-list Diversity=0.259, Catalog Coverage=0.0059) for why this was
        # added. Does not change score_candidates() or any NDCG@10/MRR evaluation.
        if diversify and len(candidates) > k:
            results = self._mmr_select(candidates, k, mmr_relevance_weight)
        else:
            results = candidates[:k]

        if len(results) < k:
            fill = self._popularity_fallback(seen | {r[0] for r in results}, k - len(results))
            results += [(i, 0.0) for i in fill]

        return [{"item_idx": i, "reason": "content-similar" if s > 0 else "popular",
                  "similarity": round(float(s), 4), "title": self.items_df.loc[i, "title"]}
                 for i, s in results]

    def score_candidates(self, user_idx: int, candidate_item_idxs: list) -> np.ndarray:
        profile_vec, is_cold = self._get_profile_vec(user_idx)
        if is_cold:
            return self.items_df.loc[candidate_item_idxs, "popularity_score"].to_numpy()
        rows = [self.item_idx_to_row[i] for i in candidate_item_idxs]
        cand_matrix = self.vectorizer.item_matrix[rows]
        if self.is_sparse:
            return (cand_matrix @ profile_vec.T).toarray().ravel()
        return cand_matrix @ profile_vec.ravel()

    def attach_cold_start(self, builder: "ColdStartProfileBuilder", diversify_fallback: bool = True) -> None:
        """Wire in cold-start handling after construction (no need to touch the call sites
        in build_recommender_for_history()). If diversify_fallback=True, also swaps the
        flat global-popularity fallback list for a category-diversified one, so different
        cold users stop seeing an identical top-N list (see cell 32: every user got the
        same three Popularity picks)."""
        self.cold_start_builder = builder
        if diversify_fallback:
            self.popularity_rank = builder.diversified_popularity_order()

    def recommend_for_new_user(self, preferred_categories: list[str] | None = None, k: int | None = None):
        """Recommend for a user who has no entry in interacted_items/profiles at all --
        i.e. a genuinely new signup, not merely a user_idx with a thin history. recommend()
        only ever sees user_idx values that already exist in the training data, so this is
        the actual user-cold-start entry point.

        If preferred_categories is given and a cold-start builder is attached, builds a
        synthetic content profile from the most popular items in those categories and
        ranks normally by content similarity. Otherwise falls back to the (diversified,
        if attached) popularity list -- same fallback path recommend() already uses."""
        k = k or CONFIG["top_k"]
        if preferred_categories and self.cold_start_builder is not None:
            profile_vec = self.cold_start_builder.onboarding_profile(preferred_categories)
            if profile_vec is not None:
                candidate_pool = CONFIG["candidate_pool"]
                dist, idx = self.vectorizer.query_topk(profile_vec, k=candidate_pool)
                dist, idx = dist[0], idx[0]
                sim = 1 - dist
                results = []
                for row, s in zip(idx, sim):
                    item_idx = int(self.row_to_item_idx[row])
                    results.append((item_idx, s))
                    if len(results) == k:
                        break
                return [
                    {"item_idx": i, "reason": "onboarding-categories", "similarity": round(float(s), 4),
                     "title": self.items_df.loc[i, "title"]}
                    for i, s in results
                ]
        fallback = (
            self.cold_start_builder.diversified_popularity_order()
            if self.cold_start_builder is not None
            else self.popularity_rank
        )
        return [
            {"item_idx": i, "reason": "popular", "similarity": 0.0, "title": self.items_df.loc[i, "title"]}
            for i in fallback[:k]
        ]

    def _new_item_input(self, item_fields: dict[str, str]):
        """Builds the input transform_new_items() actually expects for the *active*
        vectorizer. TFIDFHashingVectorizer/SVDContentVectorizer/SBERTContentVectorizer all
        take a flat content_text string (built the same way build_unweighted_content_text()
        builds every existing item's text, so a cold item is represented identically to a
        warm one); FieldWeightedTFIDFVectorizer/FieldWeightedSVDContentVectorizer instead
        need the raw per-field columns to apply their field weights correctly -- passing
        those a flat string would silently lose the field boundaries the weights depend on.

        Previously score_new_item() always passed a flat string ([content_text]), which
        worked for TFIDF/SVD/SBERT but raised inside FieldWeightedTFIDFVectorizer's
        prepare_item_frame() (which expects DataFrame columns, not a bare string) --
        i.e. it was broken for Weighted-TFIDF-TUNED and Weighted-SVD-TUNED, the two
        currently-winning model families. This dispatches on vectorizer type instead."""
        one_row = pd.DataFrame([
            {field: item_fields.get(field, "") for field in ITEM_TEXT_FIELDS}
        ])
        if isinstance(self.vectorizer, (FieldWeightedTFIDFVectorizer, FieldWeightedSVDContentVectorizer)):
            return one_row
        content_text = build_unweighted_content_text(one_row, CONFIG["description_truncate_chars"]).tolist()
        return content_text

    def score_new_item(self, item_fields: dict[str, str]) -> np.ndarray:
        """Item-cold-start path: embed a brand-new item into the already-fitted vector
        space (transform only, never fit) and return its vector. Takes the same raw
        per-field dict used everywhere else in this notebook (title/categories/features/
        description/author) rather than a single flattened string -- see _new_item_input()
        for why. Demonstrates content-based's classic advantage over collaborative
        filtering: a new book can be scored immediately from its metadata, with zero
        interactions. This only computes the vector/similarity; call register_new_item()
        to make the item actually reachable through recommend()."""
        return self.vectorizer.transform_new_items(self._new_item_input(item_fields))

    def _new_item_similarity(self, new_vector, profile_vec) -> float:
        if self.is_sparse:
            new_vector = new_vector if sp.issparse(new_vector) else sp.csr_matrix(new_vector)
            return float((new_vector @ profile_vec.T).toarray().ravel()[0])
        return float(np.asarray(new_vector).ravel() @ np.asarray(profile_vec).ravel())

    def register_new_item(
        self,
        item_idx: int,
        item_fields: dict[str, str],
        title: str | None = None,
    ) -> np.ndarray:
        """Makes a brand-new item reachable through recommend(), not just score_new_item()'s
        isolated similarity number. Computes its vector via score_new_item(), stores it in
        pending_new_items (merged into every future recommend() call's candidate pool --
        see the change there), and adds a row to items_df so title/category/author lookups
        (recommend()'s output dict, MMR's item_metadata_similarity()) work normally.

        item_idx must not collide with an existing item -- e.g. use a negative int or an
        id outside the current catalogue's range."""
        if item_idx in self.item_idx_to_row:
            raise ValueError(f"item_idx={item_idx} already exists in the fitted item index")
        vector = self.score_new_item(item_fields)
        self.pending_new_items[item_idx] = vector
        self.items_df.loc[item_idx, "title"] = title or item_fields.get("title", "")
        self.items_df.loc[item_idx, "categories"] = item_fields.get("categories", "")
        self.items_df.loc[item_idx, "author"] = item_fields.get("author", "")
        return vector


### 输出格式对齐: `recommend_as_dataframe()`

`ContentRecommender.recommend()` 本身仍然返回原来的 list-of-dict（`item_idx`/`reason`/
`similarity`/`title`）——candidate_pool消融、冷启动demo、关键词解释demo等好几处已有代码
都是按这个格式消费的，改动它的返回类型要连带重写那几处，性价比不高。

这里加一个单独的适配函数，把同样的推荐结果转换成 `Kindle_SASRec_Sequential_Recommender.ipynb`
里 `recommend_top_k()` 的输出形状：一个DataFrame，列是 `rank`（1开始的整数）、`item_idx`、`score`
（把 `similarity` 改名，对齐字段名），再merge进 `author`/`categories`/`average_rating`/
`rating_number`（对方还merge了 `title`，这里 `title` 已经在原始dict里了）。两边分数的数值含义
其实不同——对方的 `score` 是SASRec/GRU4Rec的内积logit，没有固定范围；这里的 `score` 还是content-
based的余弦相似度（0到1之间）——只对齐了表格结构和列名，没有假装这两个数字可以直接比大小。

In [17]:
def recommend_as_dataframe(
    recommender,
    user_idx: int,
    k: int | None = None,
    **recommend_kwargs,
) -> pd.DataFrame:
    """Adapts recommend()'s native list-of-dict output into the same DataFrame shape as
    the sequential notebook's recommend_top_k(): rank/item_idx/score + merged metadata.
    See the markdown note above for why this is a separate wrapper instead of changing
    recommend()'s own return type."""
    k = k or CONFIG["top_k"]
    recommendations = recommender.recommend(user_idx, k=k, **recommend_kwargs)
    frame = pd.DataFrame(recommendations)
    if frame.empty:
        columns = ["rank", "item_idx", "score", "title", "reason"]
        return pd.DataFrame(columns=columns)

    frame = frame.rename(columns={"similarity": "score"})
    frame.insert(0, "rank", np.arange(1, len(frame) + 1))

    metadata_cols = [
        col for col in ["author", "categories", "average_rating", "rating_number"]
        if col in recommender.items_df.columns
    ]
    if metadata_cols:
        frame = frame.merge(
            recommender.items_df[metadata_cols],
            left_on="item_idx",
            right_index=True,
            how="left",
        )

    ordered_cols = ["rank", "item_idx", "score", "title", *metadata_cols, "reason"]
    return frame[[col for col in ordered_cols if col in frame.columns]]


# Example (after run_all_variants() has built `recommenders`):
# recommend_as_dataframe(recommenders["Basic-TFIDF"], user_idx=0, k=10)


# Evaluation

All methods use the same one-positive-plus-100-fixed-negatives candidate sets. Candidate order is deterministically shuffled with the shared evaluation seed before stable sorting, so score ties do not systematically favour the positive item.

`Hit@10`, `Recall@10`, `Precision@10`, and `NDCG@10` are cutoff metrics. `MRR` uses the positive rank over all 101 candidates, matching the sequential notebook.


> **Alignment with the sequential notebook's evaluation protocol**
>
> `summarise_ranks` below also reports `CandidateAccuracy@10` (`(num_candidates - k + Hit@10) / num_candidates`), matching `Kindle_SASRec_Sequential_Recommender.ipynb` exactly -- this is a schema-parity addition, not a new evaluation idea, so the two teams' results tables have identical columns when combined. It is included only for compatibility with the shared reporting table; with one relevant item among 101 candidates it is far less informative than NDCG/Recall/MRR.
>
> `run_all_variants()` and the final test-lock cell both also write into `outputs/candidate_set_results.csv` -- the same filename and column schema (`model, split, Hit@10, Recall@10, Precision@10, CandidateAccuracy@10, NDCG@10, MRR, users`) the sequential notebook writes -- so a later hybrid-fusion notebook can load all three teammates' results with one `pd.read_csv` per file and no column reconciliation.


In [18]:
def rank_from_scores(
    user_idx: int,
    candidates: list[int],
    scores: np.ndarray,
    seed: int,
) -> int:
    candidates = np.asarray(candidates, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if len(candidates) != len(scores):
        raise ValueError("Candidate and score lengths differ")

    # The positive starts at candidate position zero.
    rng = np.random.default_rng(seed + int(user_idx))
    permutation = rng.permutation(len(candidates))
    shuffled_scores = scores[permutation]
    shuffled_positive_position = int(np.flatnonzero(permutation == 0)[0])
    order = np.argsort(-shuffled_scores, kind="stable")
    return int(np.flatnonzero(order == shuffled_positive_position)[0]) + 1


def summarise_ranks(
    ranks: list[int] | np.ndarray,
    k: int,
    model_name: str,
    split: str,
    num_candidates: int = 101,
) -> dict:
    ranks = np.asarray(ranks, dtype=np.int64)
    if not 0 < k < num_candidates:
        raise ValueError("k must be between 1 and num_candidates - 1")
    hits = ranks <= k
    hit_rate = float(hits.mean())
    return {
        "model": model_name,
        "split": normalise_split_name(split),
        f"Hit@{k}": hit_rate,
        f"Recall@{k}": hit_rate,
        f"Precision@{k}": hit_rate / k,
        # Schema parity with Kindle_SASRec_Sequential_Recommender.ipynb's summarise_ranks:
        # identical formula, so the combined team results table has matching columns.
        # Classifying the top-k as positive yields k predictions out of num_candidates per
        # user; with a single relevant item this is far less informative than NDCG/MRR.
        f"CandidateAccuracy@{k}": float((num_candidates - k + hit_rate) / num_candidates),
        f"NDCG@{k}": float(
            np.where(hits, 1.0 / np.log2(ranks + 1), 0.0).mean()
        ),
        "MRR": float((1.0 / ranks).mean()),
        "users": int(len(ranks)),
    }


def evaluate_ranking(
    recommender,
    positives_df,
    negatives_dict,
    k: int,
    model_name: str,
    split: str,
    seed: int | None = None,
) -> dict:
    seed = CONFIG["evaluation_seed"] if seed is None else seed
    ranks = []
    num_candidates = 101
    for row in positives_df.itertuples():
        user_idx = int(row.user_idx)
        positive = int(row.pos_idx)
        candidates = [positive, *negatives_dict[user_idx]]
        num_candidates = len(candidates)
        scores = recommender.score_candidates(user_idx, candidates)
        ranks.append(rank_from_scores(user_idx, candidates, scores, seed))
    return summarise_ranks(ranks, k, model_name, split, num_candidates=num_candidates)


def item_metadata_similarity(items_indexed: pd.DataFrame, left: int, right: int) -> float:
    left_categories = {
        value.strip()
        for value in str(items_indexed.at[left, "categories"]).split(";")
        if value.strip()
    }
    right_categories = {
        value.strip()
        for value in str(items_indexed.at[right, "categories"]).split(";")
        if value.strip()
    }
    category_similarity = len(left_categories & right_categories) / max(
        len(left_categories | right_categories), 1
    )
    left_author = str(items_indexed.at[left, "author"]).strip()
    right_author = str(items_indexed.at[right, "author"]).strip()
    author_similarity = float(bool(left_author) and left_author == right_author)
    return max(category_similarity, author_similarity)


def evaluate_catalogue_diagnostics(
    recommender,
    user_idxs,
    n_total_items: int,
    k: int,
    train_item_counts: np.ndarray,
) -> tuple[pd.DataFrame, dict]:
    """Per-user + aggregate catalogue diagnostics. Field names and (DataFrame, dict)
    return shape now match Kindle_SASRec_Sequential_Recommender.ipynb's
    recommendation_diagnostics() exactly -- novelty/intra_list_diversity/
    avg_train_popularity per user; users/catalogue_coverage/unique_recommended_items
    aggregated -- so results from both notebooks can be combined without reconciling
    column names. Previously this returned one already-averaged dict with Title-Case
    keys and a "Catalog" (not "Catalogue") spelling that matched neither teammate's
    convention, and it discarded the per-user rows entirely."""
    recommended_items = set()
    total_train_interactions = int(train_item_counts.sum())
    rows = []

    for user_idx in user_idxs:
        recommendations = recommender.recommend(int(user_idx), k=k)
        items = [int(row["item_idx"]) for row in recommendations]
        recommended_items.update(items)

        novelty = float(np.mean([
            -np.log2(
                (train_item_counts[item] + 1) / (total_train_interactions + n_total_items)
            )
            for item in items
        ]))
        pair_similarities = [
            item_metadata_similarity(recommender.items_df, left, right)
            for position, left in enumerate(items)
            for right in items[position + 1 :]
        ]
        intra_list_diversity = (
            1.0 - float(np.mean(pair_similarities)) if pair_similarities else 0.0
        )
        avg_train_popularity = float(np.mean(train_item_counts[items]))

        rows.append({
            "user_idx": int(user_idx),
            "novelty": novelty,
            "intra_list_diversity": intra_list_diversity,
            "avg_train_popularity": avg_train_popularity,
        })

    diagnostics_df = pd.DataFrame(rows)
    catalogue_metrics = {
        "users": len(user_idxs),
        "catalogue_coverage": len(recommended_items) / n_total_items,
        "unique_recommended_items": len(recommended_items),
    }
    return diagnostics_df, catalogue_metrics


def standardise_candidate_scores(scores: np.ndarray) -> np.ndarray:
    """Per-user z-score standardisation of candidate scores -- identical name and formula
    to the sequential notebook's `standardise_candidate_scores` (used there to blend
    GRU4Rec and Markov-1 scores). Kept under the same name/formula here, unused by any
    single-model evaluation above, so a later hybrid-fusion notebook can combine
    content-based, CF, and sequential candidate scores with one shared normalisation
    convention instead of three different ad hoc ones."""
    scores = np.asarray(scores, dtype=np.float64)
    std = scores.std()
    return (scores - scores.mean()) / max(std, 1e-8)


# Rooted under Drive on Colab (same COLAB_OUTPUT_ROOT as OUTPUT_DIR above) so this
# cross-team shared file also survives a runtime restart; kept as its own "outputs"
# subfolder name either way -- that's the shared convention with the CF/sequential
# notebooks, so all three teammates' candidate_set_results.csv end up at the same
# relative path name inside whichever root each of you actually uses.
SHARED_OUTPUTS_DIR = (
    Path(COLAB_OUTPUT_ROOT) / "outputs" if IN_COLAB else Path("outputs")
)


def save_candidate_set_results(
    rows_df: pd.DataFrame,
    path: Path = SHARED_OUTPUTS_DIR / "candidate_set_results.csv",
) -> None:
    """Upserts into the same outputs/candidate_set_results.csv file the sequential notebook
    already writes (model, split, Hit@10, Recall@10, Precision@10, CandidateAccuracy@10,
    NDCG@10, MRR, users) -- so all three teammates' results can be combined with a single
    pd.concat / pd.read_csv, with no column-name reconciliation. Existing (model, split)
    rows are replaced rather than duplicated, so re-running a cell updates its own rows only."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat([existing, rows_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=["model", "split"], keep="last")
    else:
        combined = rows_df
    combined.to_csv(path, index=False)


> **Result interpretation**
>
> These are sampled candidate-set metrics: one held-out positive is ranked against 100 fixed negatives per user. They are not full-catalogue Top-10 metrics.
>
> With one relevant item, `Recall@10 == Hit@10` and `Precision@10 == Hit@10 / 10`. `MRR` is computed from the full rank among all 101 candidates. Report `NDCG@10` as the pre-declared primary model-selection metric so the Content-based and Sequential experiments use the same criterion.


# Run All Variants and Compare

### 命名说明

`ContentBased-SVD` 这个名字容易被误解为"对用户-物品评分矩阵做协同过滤矩阵分解（Matrix Factorization）"。**实际上我们的SVD是对 item×TF-IDF词矩阵 做降维，属于内容特征降维，和CF方法的评分矩阵分解是两回事**，只是恰好都用了SVD这项技术。

为避免和团队里CF同学的方法混淆，**报告文字里**建议统一改用下表右列的说法（代码里的变量名/model字符串不用改，保留是为了不破坏已经跑出的历史结果的可比性，只在报告展示时做名称映射）：

In [19]:
# 报告展示用的名称映射：不改变代码里实际使用的 model 字符串（避免破坏已收集结果的可比性），
# 只在最终展示/导出报告表格时，把内部命名换成更不容易引起误解的说法。
REPORT_NAME_MAP = {
    "Popularity": "Popularity (no personalisation)",
    "Basic-TFIDF": "TFIDF-Cosine",
    "Weighted-TFIDF": "Field-Weighted TFIDF-Cosine",
    "ContentBased-SVD": "TFIDF-SVD-Cosine",
    "ContentBased-SBERT-MiniLM": "SBERT-MiniLM-Cosine",
    "ContentBased-SBERT-mpnet": "SBERT-MPNet-Cosine",
}

def to_report_names(df, name_col="model"):
    """返回一份用于报告展示的副本，把内部model名换成REPORT_NAME_MAP里更清晰的说法。"""
    df = df.copy()
    df[name_col] = df[name_col].map(lambda n: REPORT_NAME_MAP.get(n, n))
    return df

In [20]:
def build_recommender_for_history(
    vectorizer,
    items_df,
    interactions_df,
    n_users,
    item_idx_to_row,
    use_lazy_profiles,
    verified_purchase_multiplier,
):
    if use_lazy_profiles:
        profiles = LazyUserProfiles(
            interactions_df,
            vectorizer.item_matrix,
            item_idx_to_row,
            verified_purchase_multiplier,
        )
        interacted = profiles.interacted_items
    else:
        profiles, interacted = build_user_profiles_dense(
            interactions_df,
            vectorizer.item_matrix,
            n_users,
            item_idx_to_row,
            verified_purchase_multiplier,
        )
    return ContentRecommender(
        vectorizer,
        items_df,
        profiles,
        interacted,
        item_idx_to_row,
    )


def profile_history_for_split(split: str) -> pd.DataFrame:
    split = normalise_split_name(split)
    if split == "validation":
        return train_interactions
    if split == "test":
        return test_profile_interactions
    raise ValueError("Evaluation is defined only for validation or test")


def run_variant(
    name,
    vectorizer,
    items_df,
    train_interactions,
    n_users,
    n_items,
    use_lazy_profiles,
    eval_splits=("validation",),
    verified_purchase_multiplier: float | None = None,
):
    '''Fit item content once, then build the correct chronological user profile per split.'''
    del train_interactions, n_items  # histories come from profile_history_for_split()
    verified_purchase_multiplier = (
        CONFIG["verified_purchase_multiplier"]
        if verified_purchase_multiplier is None
        else verified_purchase_multiplier
    )
    eval_splits = tuple(normalise_split_name(split) for split in eval_splits)

    progress(f"[{name}] fitting item content representation...")
    item_matrix = vectorizer.fit_transform(items_df)
    item_idx_to_row = {
        int(item_idx): row
        for row, item_idx in enumerate(vectorizer.item_idx_order)
    }
    progress(f"[{name}] item_matrix shape={item_matrix.shape}", done=True)
    vectorizer.build_index(n_neighbors=CONFIG["n_neighbors_index"])

    results = []
    recommenders_by_split = {}
    for split in eval_splits:
        history = profile_history_for_split(split)
        recommender = build_recommender_for_history(
            vectorizer,
            items_df,
            history,
            n_users,
            item_idx_to_row,
            use_lazy_profiles,
            verified_purchase_multiplier,
        )
        recommenders_by_split[split] = recommender
        positives = load_eval_positives(DATA_DIR, split)
        negatives = load_eval_negatives(DATA_DIR, split)
        metrics = evaluate_ranking(
            recommender,
            positives,
            negatives,
            CONFIG["eval_k"],
            name,
            split,
        )
        results.append(metrics)
        progress(
            f"[{name}] {split}: "
            f"Hit@10={metrics['Hit@10']:.4f} "
            f"NDCG@10={metrics['NDCG@10']:.4f} "
            f"MRR={metrics['MRR']:.4f}",
            done=True,
        )

    # Return the most recent chronological recommender for diagnostics/demos.
    return recommenders_by_split[eval_splits[-1]], results


def run_all_variants(
    items_df,
    train_interactions,
    n_users,
    n_items,
    eval_splits=("validation",),
) -> dict[str, Any]:
    '''Validation-only by default. The final test is intentionally outside this function call.'''
    started = time.time()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    eval_splits = tuple(normalise_split_name(split) for split in eval_splits)

    diagnostic_rng = np.random.default_rng(CONFIG["evaluation_seed"])
    diagnostic_users = diagnostic_rng.choice(
        np.arange(n_users),
        size=min(CONFIG["diagnostic_n_users"], n_users),
        replace=False,
    )

    all_results = []
    catalogue_results = []
    catalogue_diagnostics_by_model = {}
    recommenders = {}

    def register(name, recommender, results):
        recommenders[name] = recommender
        all_results.extend(results)
        if "test" in eval_splits:
            # evaluate_catalogue_diagnostics() now returns (per_user_df, aggregate_dict) --
            # matching the sequential notebook's recommendation_diagnostics() shape. Keep
            # both: the aggregate dict feeds catalogue_results (unchanged downstream use),
            # the per-user rows are kept per model in case a caller wants the fine-grained
            # breakdown too.
            per_user_diagnostics, diagnostics = evaluate_catalogue_diagnostics(
                recommender,
                diagnostic_users,
                n_items,
                CONFIG["top_k"],
                train_item_counts,
            )
            diagnostics["model"] = name
            catalogue_results.append(diagnostics)
            catalogue_diagnostics_by_model[name] = per_user_diagnostics

    # Common train-only popularity baseline.
    popularity_history = (
        test_interacted_items if "test" in eval_splits else train_interacted_items
    )
    popularity_recommender = PopularityRecommender(items_df, popularity_history)
    popularity_results = []
    for split in eval_splits:
        metrics = evaluate_ranking(
            popularity_recommender,
            load_eval_positives(DATA_DIR, split),
            load_eval_negatives(DATA_DIR, split),
            CONFIG["eval_k"],
            "Popularity",
            split,
        )
        popularity_results.append(metrics)
        progress(
            f"[Popularity] {split}: Hit@10={metrics['Hit@10']:.4f} "
            f"NDCG@10={metrics['NDCG@10']:.4f} MRR={metrics['MRR']:.4f}",
            done=True,
        )
    register("Popularity", popularity_recommender, popularity_results)

    variant_specs = [
        (
            "Basic-TFIDF",
            TFIDFHashingVectorizer(CONFIG["basic_n_features"]),
            True,
        ),
        (
            "ContentBased-SVD",
            SVDContentVectorizer(
                CONFIG["svd_max_features"],
                CONFIG["svd_min_df"],
                CONFIG["svd_n_components"],
            ),
            False,
        ),
    ]
    if CONFIG["run_sbert"]:
        variant_specs.extend(
            [
                (
                    "ContentBased-SBERT-MiniLM",
                    SBERTContentVectorizer(
                        CONFIG["sbert_model_name"],
                        CONFIG["sbert_batch_size"],
                    ),
                    False,
                ),
                (
                    "ContentBased-SBERT-mpnet",
                    SBERTContentVectorizer(
                        "all-mpnet-base-v2",
                        CONFIG["sbert_batch_size"],
                    ),
                    False,
                ),
            ]
        )

    for name, vectorizer, use_lazy_profiles in variant_specs:
        recommender, results = run_variant(
            name,
            vectorizer,
            items_df,
            train_interactions,
            n_users,
            n_items,
            use_lazy_profiles,
            eval_splits=eval_splits,
        )
        register(name, recommender, results)

    results_df = pd.DataFrame(all_results)
    catalogue_df = pd.DataFrame(catalogue_results)

    # Cross-team schema parity: same file/columns the sequential notebook writes, so a
    # hybrid-fusion notebook can combine all three teammates' results with one pd.concat.
    save_candidate_set_results(to_report_names(results_df))

    with open(
        OUTPUT_DIR / "validation_model_comparison.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            {
                "protocol": {
                    "validation_context": "train",
                    "test_context": "train+validation",
                    "primary_metric": CONFIG["primary_metric"],
                    "evaluation_seed": CONFIG["evaluation_seed"],
                },
                "results_table": all_results,
                "elapsed_seconds": time.time() - started,
            },
            handle,
            indent=2,
        )
    progress(f"Validation comparison done in {time.time() - started:.1f}s", done=True)
    return recommenders, results_df, catalogue_df


In [21]:
# DEVELOPMENT PHASE: validation only. Do not add "test" here.
recommenders, results_df, _ = run_all_variants(
    items_df,
    train_interactions,
    n_users,
    n_items,
    eval_splits=("validation",),
)

print(
    "Validation-only model comparison. "
    f"Primary selection metric: {CONFIG['primary_metric']}"
)
display(
    to_report_names(results_df).sort_values(
        CONFIG["primary_metric"],
        ascending=False,
    )
)


[Popularity] validation: Hit@10=0.4056 NDCG@10=0.2533 MRR=0.2248                                    
[SBERT] WARNING: no GPU detected (torch.cuda.is_available()=False) -- encoding all ~81k items on CPU will be very slow (hours for mpnet). On Colab: Runtime > Change runtime type > GPU, then reconnect and rerun.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10776.36it/s]


[SBERT-all-MiniLM-L6-v2] using device=cpu                                                           
[SBERT] WARNING: no GPU detected (torch.cuda.is_available()=False) -- encoding all ~81k items on CPU will be very slow (hours for mpnet). On Colab: Runtime > Change runtime type > GPU, then reconnect and rerun.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8054.06it/s]


[SBERT-all-mpnet-base-v2] using device=cpu                                                          
[Basic-TFIDF] item_matrix shape=(81322, 32768)                                                      
[Basic-TFIDF] validation: Hit@10=0.5375 NDCG@10=0.3899 MRR=0.3614                                   
[ContentBased-SVD] item_matrix shape=(81322, 128)                                                   
[ContentBased-SVD] validation: Hit@10=0.5831 NDCG@10=0.3806 MRR=0.3344                              
[SBERT-all-MiniLM-L6-v2] loaded cached embeddings from all-MiniLM-L6-v2_e11d6f5c3a667988.embeddings.npy (delete this file to force re-encoding)
[ContentBased-SBERT-MiniLM] item_matrix shape=(81322, 384)                                          
[ContentBased-SBERT-MiniLM] validation: Hit@10=0.4401 NDCG@10=0.2801 MRR=0.2514                     
[SBERT-all-mpnet-base-v2] loaded cached embeddings from all-mpnet-base-v2_e11d6f5c3a667988.embeddings.npy (delete this file to force re-encoding)
[Co

,model,split,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users
1,TFIDF-Cosine,validation,0.537484,0.537484,0.053748,0.906312,0.389874,0.361362,64867
2,TFIDF-SVD-Cosine,validation,0.583085,0.583085,0.058309,0.906763,0.380551,0.334444,64867
4,SBERT-MPNet-Cosine,validation,0.478286,0.478286,0.047829,0.905726,0.308409,0.275441,64867
3,SBERT-MiniLM-Cosine,validation,0.440101,0.440101,0.044010,0.905348,0.280078,0.251414,64867
0,Popularity (no personalisation),validation,0.405645,0.405645,0.040565,0.905006,0.253269,0.224836,64867


### Sample recommendations per variant

In [22]:
for name, rec in recommenders.items():
    print(f"=== {name} ===")
    for u in [0, 1]:
        print(f"  user {u}:")
        for r in rec.recommend(u, k=3):
            print(f"    - [{r['reason']}] sim={r['similarity']:.3f} {r['title'][:55]}")

=== Popularity ===
  user 0:
    - [popular] sim=0.000 My Sister's Grave (Tracy Crosswhite Book 1)
    - [popular] sim=0.000 Orphan Train: A Novel
    - [popular] sim=0.000 Where the Crawdads Sing
  user 1:
    - [popular] sim=0.000 My Sister's Grave (Tracy Crosswhite Book 1)
    - [popular] sim=0.000 Orphan Train: A Novel
    - [popular] sim=0.000 Where the Crawdads Sing
=== Basic-TFIDF ===
  user 0:
    - [content-similar] sim=0.242 Atomic Habits: An Easy & Proven Way to Build Good Habit
    - [content-similar] sim=0.201 Turtles All the Way Down
    - [content-similar] sim=0.212 The Undoing Project: A Friendship That Changed Our Mind
  user 1:
    - [content-similar] sim=0.316 Deadly Associations (Book #3 in The Claudia Hershey Mys
    - [content-similar] sim=0.312 Quinn Goes to Jail (Liam Quinn Mysteries Book 8)
    - [content-similar] sim=0.286 Margaritas & Murder: A Sunny Truly Mystery (Sunny Truly
=== ContentBased-SVD ===
  user 0:
    - [content-similar] sim=0.900 Thank You for 

In [23]:
# SVD dimension tuning uses validation NDCG@10 only.
svd_dim_results = []
for dim in [32, 64, 128, 256]:
    svd_vectorizer = SVDContentVectorizer(
        CONFIG["svd_max_features"],
        CONFIG["svd_min_df"],
        dim,
    )
    _, result = run_variant(
        f"SVD-{dim}dim",
        svd_vectorizer,
        items_df,
        train_interactions,
        n_users,
        n_items,
        use_lazy_profiles=False,
        eval_splits=("validation",),
    )
    row = result[0]
    row["svd_dim"] = dim
    svd_dim_results.append(row)

svd_dim_df = pd.DataFrame(svd_dim_results)
best_svd_row = svd_dim_df.loc[
    svd_dim_df[CONFIG["primary_metric"]].idxmax()
]
BEST_SVD_DIM = int(best_svd_row["svd_dim"])
CONFIG["svd_n_components"] = BEST_SVD_DIM

print(
    f"Selected SVD dimension={BEST_SVD_DIM} using "
    f"validation {CONFIG['primary_metric']}="
    f"{best_svd_row[CONFIG['primary_metric']]:.6f}"
)
display(svd_dim_df.sort_values(CONFIG["primary_metric"], ascending=False))


[SVD-32dim] item_matrix shape=(81322, 32)                                                           
[SVD-32dim] validation: Hit@10=0.5749 NDCG@10=0.3544 MRR=0.3042                                     
[SVD-64dim] item_matrix shape=(81322, 64)                                                           
[SVD-64dim] validation: Hit@10=0.5835 NDCG@10=0.3722 MRR=0.3239                                     
[SVD-128dim] item_matrix shape=(81322, 128)                                                         
[SVD-128dim] validation: Hit@10=0.5831 NDCG@10=0.3806 MRR=0.3344                                    
[SVD-256dim] item_matrix shape=(81322, 256)                                                         
[SVD-256dim] validation: Hit@10=0.5745 NDCG@10=0.3857 MRR=0.3438                                    
Selected SVD dimension=256 using validation NDCG@10=0.385707


,model,split,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users,svd_dim
3,SVD-256dim,validation,0.574514,0.574514,0.057451,0.906678,0.385707,0.343840,64867,256
2,SVD-128dim,validation,0.583085,0.583085,0.058309,0.906763,0.380551,0.334444,64867,128
1,SVD-64dim,validation,0.583486,0.583486,0.058349,0.906767,0.372167,0.323866,64867,64
0,SVD-32dim,validation,0.574930,0.574930,0.057493,0.906682,0.354405,0.304214,64867,32


In [24]:
# No test evaluation here. The selected dimension is carried into subsequent
# validation-only ablations and the single final test cell.
assert CONFIG["svd_n_components"] == BEST_SVD_DIM
print(
    f"Locked SVD dimension from validation: {BEST_SVD_DIM}. "
    "Test remains unevaluated in this revised workflow."
)


Locked SVD dimension from validation: 256. Test remains unevaluated in this revised workflow.


## 消融实验 7：Author 字段是否有用

之前 `content_text` 构造时读取了 `author` 字段并做了缺失值填充，但没有真正拼进最终的内容文本里。这里对比"加author"和"不加author"两个版本在val集上的效果，用SVD变体做载体（Basic/SBERT同理，改一下调用对象即可）。

**注意**：这个实验假设 `items_df` / `train_interactions` / `n_users` / `n_items` 已经在全局作用域里——它们在notebook最前面的"Data Loading"部分加载一次，之后全程复用，不需要在这里重新加载。如果是从头 Restart & Run All，正常顺序执行到这里就已经具备。

In [25]:
def build_content_text(items_df, description_truncate_chars, include_author=False):
    truncated_description = items_df["description"].str.slice(0, description_truncate_chars)
    content = (
        (items_df["title"] + " ") * 2
        + (items_df["categories"].str.replace(";", " ", regex=False) + " ") * 2
    )
    if include_author:
        content = content + items_df["author"] + " "
    content = content + items_df["features"] + " " + truncated_description
    return content.str.strip()


items_df_no_author = items_df.copy()
items_df_with_author = items_df.copy()
items_df_with_author["content_text"] = build_content_text(
    items_df_with_author,
    CONFIG["description_truncate_chars"],
    include_author=True,
)

author_ablation_results = []
for label, frame in [
    ("no-author", items_df_no_author),
    ("with-author", items_df_with_author),
]:
    vectorizer = SVDContentVectorizer(
        CONFIG["svd_max_features"],
        CONFIG["svd_min_df"],
        BEST_SVD_DIM,
    )
    _, result = run_variant(
        f"SVD-{label}",
        vectorizer,
        frame,
        train_interactions,
        n_users,
        n_items,
        use_lazy_profiles=False,
        eval_splits=("validation",),
    )
    result[0]["variant"] = label
    author_ablation_results.append(result[0])

author_ablation_df = pd.DataFrame(author_ablation_results)
best_author_row = author_ablation_df.loc[
    author_ablation_df[CONFIG["primary_metric"]].idxmax()
]
BEST_INCLUDE_AUTHOR = best_author_row["variant"] == "with-author"

print(
    f"Selected include_author={BEST_INCLUDE_AUTHOR} using "
    f"validation {CONFIG['primary_metric']}="
    f"{best_author_row[CONFIG['primary_metric']]:.6f}"
)
display(
    author_ablation_df[
        ["variant", "Hit@10", "NDCG@10", "MRR"]
    ].sort_values(CONFIG["primary_metric"], ascending=False)
)


[SVD-no-author] item_matrix shape=(81322, 256)                                                      
[SVD-no-author] validation: Hit@10=0.5745 NDCG@10=0.3857 MRR=0.3438                                 
[SVD-with-author] item_matrix shape=(81322, 256)                                                    
[SVD-with-author] validation: Hit@10=0.5895 NDCG@10=0.3974 MRR=0.3541                               
Selected include_author=True using validation NDCG@10=0.397368


,variant,Hit@10,NDCG@10,MRR
1,with-author,0.589499,0.397368,0.354133
0,no-author,0.574514,0.385707,0.343840


## 消融实验 8：candidate_pool 大小 vs Popularity Fallback 触发率

`recommend()` 在候选池过滤掉用户已读的书之后,如果凑不够k=10个,会用热门书补齐(`reason="popular"`)。这里测量不同 `candidate_pool` 大小下,这种"被迫用热门书补齐"的情况发生得多不多。

**重要**：`candidate_pool` 是查询时参数,不需要重新拟合向量化器/重新构建画像——直接复用已经跑好的 `recommenders["ContentBased-SVD"]` 换参数查询即可,不会很慢。

**解读前提**：本数据集里 `cold_start_users=0`（之前验证过val/test里没有真正零训练交互的用户）,所以下面测出的"popular"理由**几乎全部**来自candidate_pool不够用触发的fallback,而不是冷启动分支——这两种"popular"来源在这份数据上是可以互相区分开的。

In [26]:
def measure_fallback_rate(recommender, user_idxs, k=10, candidate_pool=50):
    total_popular, total_recs, users_with_fallback = 0, 0, 0
    for user in user_idxs:
        recommendations = recommender.recommend(
            int(user),
            k=k,
            candidate_pool=candidate_pool,
        )
        n_popular = sum(
            row["reason"] == "popular"
            for row in recommendations
        )
        total_popular += n_popular
        total_recs += len(recommendations)
        users_with_fallback += int(n_popular > 0)
    return {
        "candidate_pool": candidate_pool,
        "avg_popular_items_per_user": total_popular / len(user_idxs),
        "pct_users_with_any_fallback": users_with_fallback / len(user_idxs),
        "pct_recs_that_are_fallback": total_popular / total_recs,
    }


diagnostic_rng = np.random.default_rng(CONFIG["evaluation_seed"])
sample_users_for_pool_test = diagnostic_rng.choice(
    np.arange(n_users),
    size=min(CONFIG["diagnostic_n_users"], n_users),
    replace=False,
)

# This diagnoses retrieval fallback only; it is not a model-selection experiment.
pool_ablation_results = []
for pool in [5, 10, 20, 50]:
    stats = measure_fallback_rate(
        recommenders["ContentBased-SVD"],
        sample_users_for_pool_test,
        k=10,
        candidate_pool=pool,
    )
    pool_ablation_results.append(stats)

pool_ablation_df = pd.DataFrame(pool_ablation_results)
display(pool_ablation_df)


,candidate_pool,avg_popular_items_per_user,pct_users_with_any_fallback,pct_recs_that_are_fallback
0,5,6.1998,1.0000,0.61998
1,10,1.5718,0.7684,0.15718
2,20,0.0066,0.0020,0.00066
3,50,0.0000,0.0000,0.00000


## 消融实验 9：verified_purchase_multiplier 是否真的有用

`build_user_profiles_dense()` 里用户画像的加权公式是：
`weight = (rating/5.0) * (verified_purchase_multiplier if verified_purchase else 1.0)`

`verified_purchase_multiplier=1.2` 是我们凭经验设的（"已验证购买的交互更可信"），一直没有真实数据验证过。这里对比 1.0（完全不区分verified，作为对照组）/ 1.2（当前默认）/ 1.5 / 2.0 四组在val集上的效果。

**做法**：这个参数是 `build_user_profiles_dense()` 内部直接从 `CONFIG["verified_purchase_multiplier"]` 现读的（不是函数参数），所以跟SVD维度消融一样,循环里改CONFIG值、重新跑一次 `run_variant()` 即可,不需要改任何函数定义。

**跑完记得把 `CONFIG["verified_purchase_multiplier"]` 改回你最终选定的值**（比如1.2或者实验里表现最好的那个）,不然会影响你notebook后面其它cell（比如最终test评估）用到的配置。

In [27]:
# Verified-purchase weighting is tuned on validation only, using the already-selected
# SVD dimension and author setting.
verified_items_df = items_df.copy()
verified_items_df["content_text"] = build_content_text(
    verified_items_df,
    CONFIG["description_truncate_chars"],
    include_author=BEST_INCLUDE_AUTHOR,
)

verified_multiplier_results = []
for multiplier in [1.0, 1.2, 1.5, 2.0]:
    vectorizer = SVDContentVectorizer(
        CONFIG["svd_max_features"],
        CONFIG["svd_min_df"],
        BEST_SVD_DIM,
    )
    _, result = run_variant(
        f"SVD-verified{multiplier}",
        vectorizer,
        verified_items_df,
        train_interactions,
        n_users,
        n_items,
        use_lazy_profiles=False,
        eval_splits=("validation",),
        verified_purchase_multiplier=multiplier,
    )
    result[0]["verified_multiplier"] = multiplier
    verified_multiplier_results.append(result[0])

verified_multiplier_df = pd.DataFrame(verified_multiplier_results)
best_verified_row = verified_multiplier_df.loc[
    verified_multiplier_df[CONFIG["primary_metric"]].idxmax()
]
BEST_VERIFIED_MULTIPLIER = float(
    best_verified_row["verified_multiplier"]
)
CONFIG["verified_purchase_multiplier"] = BEST_VERIFIED_MULTIPLIER

print(
    f"Selected verified_purchase_multiplier={BEST_VERIFIED_MULTIPLIER} "
    f"using validation {CONFIG['primary_metric']}="
    f"{best_verified_row[CONFIG['primary_metric']]:.6f}"
)
display(
    verified_multiplier_df[
        ["verified_multiplier", "Hit@10", "NDCG@10", "MRR"]
    ].sort_values(CONFIG["primary_metric"], ascending=False)
)


[SVD-verified1.0] item_matrix shape=(81322, 256)                                                    
[SVD-verified1.0] validation: Hit@10=0.5895 NDCG@10=0.3974 MRR=0.3541                               
[SVD-verified1.2] item_matrix shape=(81322, 256)                                                    
[SVD-verified1.2] validation: Hit@10=0.5890 NDCG@10=0.3968 MRR=0.3535                               
[SVD-verified1.5] item_matrix shape=(81322, 256)                                                    
[SVD-verified1.5] validation: Hit@10=0.5877 NDCG@10=0.3956 MRR=0.3524                               
[SVD-verified2.0] item_matrix shape=(81322, 256)                                                    
[SVD-verified2.0] validation: Hit@10=0.5857 NDCG@10=0.3935 MRR=0.3504                               
Selected verified_purchase_multiplier=1.0 using validation NDCG@10=0.397368


,verified_multiplier,Hit@10,NDCG@10,MRR
0,1.0,0.589499,0.397368,0.354133
1,1.2,0.589005,0.396770,0.353506
2,1.5,0.587726,0.395571,0.352383
3,2.0,0.585675,0.393538,0.350428


## 消融实验 10：Weighted-TFIDF 字段权重搜索

Adopted from `content_based_three_variants_weighted_tfidf_aligned.ipynb`. Now that
`content_text` (used by Basic-TFIDF/SVD/SBERT) is genuinely unweighted, this ablation asks
the question the old hardcoded `title*2 + categories*2` never actually tested: does
upweighting title/categories help, and which combination works best? Uses
`FieldWeightedTFIDFVectorizer` directly on the raw fields (not `content_text`), same hashing
dimension as Basic-TFIDF, selected on validation `NDCG@10` only -- same discipline as the SVD
dimension search above. The grid below is the full 14-point search from the reference notebook (restored from an earlier 8-point subset that was trimmed only for a quick runtime check). Every candidate here also gets bigrams + sublinear tf on top (see the updated `FieldWeightedTFIDFVectorizer` note above), so this ablation is now searching field weights *given* the improved text representation, not the plain-unigram one the first version of this ablation used.


In [28]:
weighted_tfidf_tuning_rows = []
for label, field_weights in CONFIG["weighted_field_weight_candidates"].items():
    _, result = run_variant(
        f"Weighted-TFIDF-{label}",
        FieldWeightedTFIDFVectorizer(
            CONFIG["basic_n_features"],
            field_weights,
            CONFIG["description_truncate_chars"],
            ngram_range=CONFIG["weighted_tfidf_ngram_range"],
            sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
        ),
        items_df,
        train_interactions,
        n_users,
        n_items,
        use_lazy_profiles=True,
        eval_splits=("validation",),
    )
    row = result[0]
    row["weight_label"] = label
    row["field_weights"] = json.dumps(field_weights, sort_keys=True)
    weighted_tfidf_tuning_rows.append(row)

weighted_tfidf_tuning_df = pd.DataFrame(weighted_tfidf_tuning_rows)
best_weighted_tfidf_row = weighted_tfidf_tuning_df.loc[
    weighted_tfidf_tuning_df[CONFIG["primary_metric"]].idxmax()
]
BEST_WEIGHT_LABEL = str(best_weighted_tfidf_row["weight_label"])
BEST_WEIGHTED_FIELD_WEIGHTS = dict(
    CONFIG["weighted_field_weight_candidates"][BEST_WEIGHT_LABEL]
)

print(
    f"Selected Weighted-TFIDF configuration={BEST_WEIGHT_LABEL} using validation "
    f"{CONFIG['primary_metric']}={best_weighted_tfidf_row[CONFIG['primary_metric']]:.6f}"
)
display(
    weighted_tfidf_tuning_df[
        ["weight_label", "Hit@10", "NDCG@10", "MRR", "field_weights"]
    ].sort_values(CONFIG["primary_metric"], ascending=False)
)


[Weighted-TFIDF-title2_categories2] item_matrix shape=(81322, 32768)                                
[Weighted-TFIDF-title2_categories2] validation: Hit@10=0.5845 NDCG@10=0.4293 MRR=0.3972             
[Weighted-TFIDF-title3_categories2] item_matrix shape=(81322, 32768)                                
[Weighted-TFIDF-title3_categories2] validation: Hit@10=0.5881 NDCG@10=0.4326 MRR=0.4001             
[Weighted-TFIDF-title2_categories3] item_matrix shape=(81322, 32768)                                
[Weighted-TFIDF-title2_categories3] validation: Hit@10=0.5904 NDCG@10=0.4312 MRR=0.3978             
[Weighted-TFIDF-title2_categories2_features1_5] item_matrix shape=(81322, 32768)                    
[Weighted-TFIDF-title2_categories2_features1_5] validation: Hit@10=0.5756 NDCG@10=0.4217 MRR=0.3900 
[Weighted-TFIDF-title2_categories2_5] item_matrix shape=(81322, 32768)                              
[Weighted-TFIDF-title2_categories2_5] validation: Hit@10=0.5885 NDCG@10=0.4308 MRR=0.3978  

,weight_label,Hit@10,NDCG@10,MRR,field_weights
13,title2_75_categories3,0.593152,0.434093,0.400600,"{""author"": 0.0, ""categories"": 3.0, ""descriptio..."
12,title2_5_categories3_25,0.593507,0.433432,0.399660,"{""author"": 0.0, ""categories"": 3.25, ""descripti..."
8,title2_5_categories3,0.592628,0.433400,0.399878,"{""author"": 0.0, ""categories"": 3.0, ""descriptio..."
11,title2_5_categories2_75,0.591965,0.433367,0.400014,"{""author"": 0.0, ""categories"": 2.75, ""descripti..."
1,title3_categories2,0.588111,0.432551,0.400107,"{""author"": 0.0, ""categories"": 2.0, ""descriptio..."
10,title2_25_categories3,0.591503,0.432377,0.398953,"{""author"": 0.0, ""categories"": 3.0, ""descriptio..."
9,title2_25_categories2_75,0.590855,0.432288,0.399001,"{""author"": 0.0, ""categories"": 2.75, ""descripti..."
2,title2_categories3,0.590439,0.431244,0.397839,"{""author"": 0.0, ""categories"": 3.0, ""descriptio..."
5,title2_categories3_5,0.591718,0.431111,0.397294,"{""author"": 0.0, ""categories"": 3.5, ""descriptio..."
4,title2_categories2_5,0.588450,0.430806,0.397847,"{""author"": 0.0, ""categories"": 2.5, ""descriptio..."


### 消融实验11: Field-Weighted TF-IDF + SVD 降维叠加测试

用上面刚选出的 `BEST_WEIGHTED_FIELD_WEIGHTS`（已经是验证集NDCG@10最优的字段权重组合），
额外叠加SVD降维，在 `weighted_svd_dim_candidates` 网格上搜索。对比对象是**不降维**的
Weighted-TFIDF-TUNED本身（`best_weighted_tfidf_row`）——如果所有降维后的NDCG@10都比它低，
说明降维在这个字段加权表示上依然是净损耗（呼应`TFIDF-SVD-TUNED`在普通TF-IDF上的结论）；
如果有一档持平或更好，则说明降维的去噪收益在信号更强的表示上终于能盖过信息损失，值得
作为新的候选家族进入最终锁定测试。

In [29]:
weighted_svd_tuning_rows = []
for n_components in CONFIG["weighted_svd_dim_candidates"]:
    _, result = run_variant(
        f"Weighted-SVD-dim{n_components}",
        FieldWeightedSVDContentVectorizer(
            CONFIG["basic_n_features"],
            BEST_WEIGHTED_FIELD_WEIGHTS,
            CONFIG["description_truncate_chars"],
            n_components,
            ngram_range=CONFIG["weighted_tfidf_ngram_range"],
            sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
        ),
        items_df,
        train_interactions,
        n_users,
        n_items,
        use_lazy_profiles=False,
        eval_splits=("validation",),
    )
    row = result[0]
    row["n_components"] = n_components
    weighted_svd_tuning_rows.append(row)

weighted_svd_tuning_df = pd.DataFrame(weighted_svd_tuning_rows)
best_weighted_svd_row = weighted_svd_tuning_df.loc[
    weighted_svd_tuning_df[CONFIG["primary_metric"]].idxmax()
]
BEST_WEIGHTED_SVD_DIM = int(best_weighted_svd_row["n_components"])

no_reduction_score = float(best_weighted_tfidf_row[CONFIG["primary_metric"]])
best_reduced_score = float(best_weighted_svd_row[CONFIG["primary_metric"]])
verdict = (
    "SVD HELPS on top of field-weighting"
    if best_reduced_score > no_reduction_score
    else "SVD still hurts, even on the field-weighted representation"
)

print(
    f"Best SVD dim on field-weighted counts={BEST_WEIGHTED_SVD_DIM} using validation "
    f"{CONFIG['primary_metric']}={best_reduced_score:.6f} "
    f"(no-reduction Weighted-TFIDF-TUNED={no_reduction_score:.6f}) -> {verdict}"
)
display(
    weighted_svd_tuning_df[["n_components", "Hit@10", "NDCG@10", "MRR"]].sort_values(
        CONFIG["primary_metric"], ascending=False
    )
)


[Weighted-SVD-dim128] item_matrix shape=(81322, 128)                                                
[Weighted-SVD-dim128] validation: Hit@10=0.5919 NDCG@10=0.3863 MRR=0.3392                           
[Weighted-SVD-dim256] item_matrix shape=(81322, 256)                                                
[Weighted-SVD-dim256] validation: Hit@10=0.6005 NDCG@10=0.4008 MRR=0.3550                           
[Weighted-SVD-dim512] item_matrix shape=(81322, 512)                                                
[Weighted-SVD-dim512] validation: Hit@10=0.6065 NDCG@10=0.4141 MRR=0.3702                           
Best SVD dim on field-weighted counts=512 using validation NDCG@10=0.414140 (no-reduction Weighted-TFIDF-TUNED=0.434093) -> SVD still hurts, even on the field-weighted representation


,n_components,Hit@10,NDCG@10,MRR
2,512,0.606503,0.414140,0.370217
1,256,0.600490,0.400802,0.355050
0,128,0.591934,0.386322,0.339180


## 推荐理由细化

当前 `recommend()` 返回的 `reason` 只有 `"content-similar"` / `"popular"` 两种,对用户界面来说太笼统。这里加两层更具体的解释,不影响原有排序逻辑(纯粹是"打分排序"和"生成解释文字"这两件事解耦):

1. **类目重合解释**(通用,任何变体都能用)——比较候选书的categories和用户过去读过的书里最常见的categories,生成"Because you liked books in {category}"这种句子。不依赖向量内部结构,只用metadata,所以对Basic/SVD/SBERT都适用。
2. **关键词重合解释**(只有保留原始TF-IDF词表的路径能做)——`SVDContentVectorizer`在SVD降维前会先算一次完整TF-IDF,这里额外存一份降维前的原始TF-IDF向量(只用来生成解释文字,不影响用来打分排序的降维向量),取用户画像和候选书之间TF-IDF权重乘积最高的几个词作为"共同关键词"。Basic(哈希trick)和SBERT(向量维度无法映射回词)做不到这一层,只能退回类目解释。

**已直接合并进前面 "Variant 1: TF-IDF + TruncatedSVD" 那个分节的 `SVDContentVectorizer` 类定义里**（不再是追加在notebook末尾的重复定义），避免"先用旧版类构建了recommenders、后面才重新定义新版类"导致新版定义不生效的顺序问题。

**用法**：只要正常从头到尾按顺序跑这个notebook（或者从头 Restart & Run All），`run_all_variants()` 构建出来的 `recommenders["ContentBased-SVD"]` 自然就带着 `raw_tfidf_matrix`，下面的解释函数可以直接用，不需要额外重跑。

In [30]:
# 只用于"生成解释文字"这一步的噪声词过滤，不改动向量化器本身、不影响任何已收集的
# 消融实验/评估结果。真实数据里description混入了一些平台样板文字(比如"Amazon
# Bestseller"这类)，这些词TF-IDF权重可能不低但对用户来说毫无解释价值，过滤掉即可。
EXPLANATION_STOPWORDS = {
    "amazon", "kindle", "book", "books", "author", "authors", "edition",
    "editions", "bestseller", "bestselling", "novel", "novels", "press",
    "publisher", "publishing", "copyright", "isbn", "series", "read",
    "reading", "story", "stories",
}


def explain_by_category(items_df, interacted_items, user_idx, candidate_item_idx, top_n=1):
    """通用解释：候选书的categories 与用户过去读过的书最常见的categories 的重合。"""
    seen = interacted_items.get(user_idx, set())
    if not seen:
        return None
    items_indexed = items_df.set_index("item_idx")
    user_categories = []
    for i in seen:
        if i in items_indexed.index:
            cats = [c.strip() for c in items_indexed.loc[i, "categories"].split(";") if c.strip()]
            user_categories.extend(cats)
    if not user_categories:
        return None
    from collections import Counter
    top_user_cats = {c for c, _ in Counter(user_categories).most_common(5)}

    cand_cats = [c.strip() for c in items_indexed.loc[candidate_item_idx, "categories"].split(";") if c.strip()]
    overlap = [c for c in cand_cats if c in top_user_cats]
    if not overlap:
        return None
    return f"Because you liked books in {', '.join(overlap[:top_n])}"


def explain_by_keywords(vectorizer, interacted_items, user_idx, candidate_item_idx, item_idx_to_row, top_n=3):
    """关键词解释：只对保留了原始TF-IDF词表的向量化器有效(目前是SVDContentVectorizer)。
    候选词按TF-IDF乘积权重降序取前几个,同时过滤掉 EXPLANATION_STOPWORDS 里的平台噪声词。"""
    if getattr(vectorizer, "raw_tfidf_matrix", None) is None:
        return None
    seen = interacted_items.get(user_idx, set())
    if not seen:
        return None

    rows = [item_idx_to_row[i] for i in seen if i in item_idx_to_row]
    if not rows:
        return None
    user_raw_profile = vectorizer.raw_tfidf_matrix[rows].sum(axis=0)
    user_raw_profile = np.asarray(user_raw_profile).ravel()

    cand_row = item_idx_to_row[candidate_item_idx]
    cand_vec = vectorizer.raw_tfidf_matrix[cand_row].toarray().ravel()

    shared_weight = user_raw_profile * cand_vec
    order = np.argsort(-shared_weight)
    top_words = []
    for i in order:
        if shared_weight[i] <= 0:
            break
        word = vectorizer.feature_names[i]
        if word in EXPLANATION_STOPWORDS:
            continue
        top_words.append(word)
        if len(top_words) == top_n:
            break
    if not top_words:
        return None
    return f"Recommended because it matches your interests in: {', '.join(top_words)}"


def explain_recommendation(vectorizer, items_df, interacted_items, user_idx, candidate_item_idx, item_idx_to_row):
    """优先用关键词解释(更具体)，拿不到就退回类目解释，两个都没有就用默认的相似度理由。"""
    kw = explain_by_keywords(vectorizer, interacted_items, user_idx, candidate_item_idx, item_idx_to_row)
    if kw:
        return kw
    cat = explain_by_category(items_df, interacted_items, user_idx, candidate_item_idx)
    if cat:
        return cat
    return "Similar to books you have read"

### 效果预览：跟原来的"content-similar"理由对比

In [31]:
# 用SVD变体演示效果对比(需要先跑过 run_all_variants()，拿到 recommenders["ContentBased-SVD"] 和 items_df 等变量)
svd_recommender = recommenders["ContentBased-SVD"]
svd_vectorizer = svd_recommender.vectorizer

for u in [0, 1, 2]:
    print(f"user {u}:")
    for r in svd_recommender.recommend(u, k=3):
        detailed_reason = explain_recommendation(
            svd_vectorizer, items_df, svd_recommender.interacted_items,
            u, r["item_idx"], svd_recommender.item_idx_to_row,
        )
        print(f"    原始: [{r['reason']}] {r['title'][:50]}")
        print(f"    细化: {detailed_reason}")

user 0:
    原始: [content-similar] Thank You for Being Late: An Optimist's Guide to T
    细化: Recommended because it matches your interests in: times, new, year
    原始: [content-similar] Weapons of Math Destruction: How Big Data Increase
    细化: Recommended because it matches your interests in: times, financial, sciences
    原始: [content-similar] To Pixar And Beyond: My Unlikely Journey with Stev
    细化: Recommended because it matches your interests in: business, times, new
user 1:
    原始: [content-similar] In A Bind: A Private Investigator Crime and Suspen
    细化: Recommended because it matches your interests in: mystery, cal, suspense
    原始: [content-similar] This Doesn't Happen In The Movies (The Reed Fergus
    细化: Recommended because it matches your interests in: mystery, investigator, detective
    原始: [content-similar] Loose Ends: A Private Investigator Crime and Suspe
    细化: Recommended because it matches your interests in: mystery, cal, suspense
user 2:
    原始: [content-simil

# Cold-Start Handling

This dataset has `cold_start_users=0` -- the 5-core filter used during preprocessing removes
any user or item with fewer than 5 positive interactions, so every user and item already has
some history by construction. That means the two things production content-based systems
have to deal with -- a brand-new user with no reading history, and a brand-new book with no
interactions yet -- are never actually exercised anywhere above, including in `recommend()`'s
existing `is_cold` fallback branch (which is currently dead code on this data).

This section adds:

1. **User cold-start** (`ColdStartProfileBuilder.onboarding_profile`) -- lets a genuinely new
   user get real content-based recommendations from a handful of stated category preferences,
   instead of only ever seeing a flat, identical popularity list.
2. **Item cold-start** (`transform_new_items` / `score_new_item`) -- lets a brand-new book be
   scored the moment its metadata exists, with zero retraining and zero interactions. This is
   the classic advantage content-based has over collaborative filtering, which needs
   interaction data before it can place a new item anywhere in its embedding space.
   `score_new_item()` now takes a raw per-field dict instead of a flat string, because the
   flat-string version silently broke for `FieldWeightedTFIDFVectorizer`/
   `FieldWeightedSVDContentVectorizer` (i.e. `Weighted-TFIDF-TUNED`/`Weighted-SVD-TUNED`,
   the currently-winning families) -- see `_new_item_input()`'s docstring below. A new
   `register_new_item()` also makes a scored item actually reachable through `recommend()`
   itself (merged into the candidate pool at serving time), not just scoreable in isolation.
3. **A simulated evaluation** -- since this dataset has no real cold users, the only way to
   check whether the new user-cold-start path is actually better than the old flat fallback is
   to pretend a sample of users have no history and compare `NDCG@10` on their real held-out
   positive, using the same 1-positive+100-negative protocol as the main results table.


In [32]:
class ColdStartProfileBuilder:
    """Builds recommendations for users the model has never seen a single interaction for.

    Two independent capabilities, both computed once from the fitted item vectors so they
    add no cost to the main pipeline:
      - `diversified_popularity_order()`: round-robins the most popular items across
        categories, so the fallback list used when there is truly no signal at all isn't
        dominated by whichever single genre happens to have the most bestsellers.
      - `onboarding_profile(preferred_categories)`: builds a synthetic user profile from the
        most popular items in a handful of stated categories, in the *same* vector space as
        the fitted vectorizer -- so it can be scored with the exact same cosine-similarity
        code path as a normal warm-user profile, no special-casing downstream.
    """

    def __init__(self, items_df, item_matrix, item_idx_to_row, is_sparse: bool):
        self.items_indexed = items_df.set_index("item_idx")
        self.item_matrix = item_matrix
        self.item_idx_to_row = item_idx_to_row
        self.is_sparse = is_sparse

        exploded = items_df.assign(category_list=items_df["categories"].str.split(";")).explode(
            "category_list"
        )
        exploded["category_list"] = exploded["category_list"].str.strip()
        exploded = exploded[exploded["category_list"] != ""]
        self.category_rank = (
            exploded.sort_values("popularity_score", ascending=False)
            .groupby("category_list")["item_idx"]
            .apply(list)
            .to_dict()
        )
        self.global_rank = (
            items_df.sort_values("popularity_score", ascending=False)["item_idx"].tolist()
        )

    def diversified_popularity_order(self, top_categories: int = 20) -> list[int]:
        """Round-robins the most popular items across the `top_categories` largest
        categories, instead of one flat global ranking dominated by whichever genre has
        the deepest catalogue (Literature/Fiction here -- see the category counts above)."""
        top_cats = sorted(
            self.category_rank, key=lambda c: len(self.category_rank[c]), reverse=True
        )[:top_categories]
        pools = [iter(self.category_rank[c]) for c in top_cats]
        seen: set[int] = set()
        ordered: list[int] = []
        while pools:
            next_pools = []
            for pool in pools:
                for item in pool:
                    if item not in seen:
                        seen.add(item)
                        ordered.append(item)
                        next_pools.append(pool)
                        break
            pools = next_pools
        for item in self.global_rank:
            if item not in seen:
                seen.add(item)
                ordered.append(item)
        return ordered

    def onboarding_profile(self, preferred_categories: list[str], n_seed_items: int = 10):
        """Synthetic profile = normalised sum of the top-`n_seed_items` most popular books
        in each stated category. Returns None if none of the categories match anything in
        the catalogue, so callers can fall back to `diversified_popularity_order()`."""
        seed_items: list[int] = []
        for category in preferred_categories:
            seed_items.extend(self.category_rank.get(category, [])[:n_seed_items])
        seed_items = [i for i in dict.fromkeys(seed_items) if i in self.item_idx_to_row]
        if not seed_items:
            return None

        rows = [self.item_idx_to_row[i] for i in seed_items]
        vecs = self.item_matrix[rows]
        if self.is_sparse:
            profile = sp.csr_matrix(vecs.sum(axis=0))
            norm = np.sqrt(profile.multiply(profile).sum())
        else:
            profile = vecs.sum(axis=0, keepdims=True)
            norm = np.linalg.norm(profile)
        if norm > 0:
            profile = profile / norm
        return profile


### Demo: item cold-start and diversified / onboarding user cold-start

Uses the already-fitted `ContentBased-SVD` recommender from `run_all_variants()` above --
no refitting, no index rebuild.


In [33]:
svd_rec = recommenders["ContentBased-SVD"]

new_book_fields = {
    "title": "The Quiet After Midnight",
    "categories": "Mystery;Thriller;Suspense",
    "features": "",
    "description": (
        "A detective investigates a string of murders in a small coastal town, "
        "uncovering secrets that connect victims to a decades-old unsolved case."
    ),
    "author": "",
}

print("=== Item cold-start: scoring a brand-new book with zero interactions ===")
# score_new_item() now takes the same raw per-field dict used everywhere else in this
# notebook (title/categories/features/description/author), not a single flattened string --
# see _new_item_input()'s docstring on ContentRecommender for why: FieldWeightedTFIDFVectorizer
# and FieldWeightedSVDContentVectorizer need the field boundaries to weight correctly, and a
# flat string silently lost that information.
new_item_vec = svd_rec.score_new_item(new_book_fields)
user1_profile, _ = svd_rec._get_profile_vec(1)  # user 1's real recs were mystery/PI novels (see earlier sample)
similarity_to_user1 = float(new_item_vec.ravel() @ user1_profile.ravel())
print(f"cosine similarity of the brand-new book to user 1's profile: {similarity_to_user1:.3f}")
print("(computed with zero retraining and zero interactions for the new book)")

print()
print("=== Item cold-start compatibility check: Weighted-TFIDF family ===")
# score_new_item() used to crash here with a TypeError: FieldWeightedTFIDFVectorizer's
# transform_new_items() expects a DataFrame with per-field columns, and the old code
# always passed a flat string regardless of vectorizer type. BEST_WEIGHTED_FIELD_WEIGHTS
# comes from the field-weight search ablation earlier in this notebook (already run by
# this point in a top-to-bottom execution).
weighted_probe_vectorizer = FieldWeightedTFIDFVectorizer(
    CONFIG["basic_n_features"],
    BEST_WEIGHTED_FIELD_WEIGHTS,
    CONFIG["description_truncate_chars"],
    ngram_range=CONFIG["weighted_tfidf_ngram_range"],
    sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
)
weighted_probe_vectorizer.fit_transform(items_df)
weighted_probe_vectorizer.build_index(n_neighbors=CONFIG["n_neighbors_index"])
weighted_probe_item_idx_to_row = {
    int(item_idx): row for row, item_idx in enumerate(weighted_probe_vectorizer.item_idx_order)
}
weighted_probe_recommender = build_recommender_for_history(
    weighted_probe_vectorizer,
    items_df,
    train_interactions,
    n_users,
    weighted_probe_item_idx_to_row,
    use_lazy_profiles=True,
    verified_purchase_multiplier=CONFIG["verified_purchase_multiplier"],
)
weighted_new_item_vec = weighted_probe_recommender.score_new_item(new_book_fields)
print(
    f"Weighted-TFIDF score_new_item() succeeded, vector shape={weighted_new_item_vec.shape} "
    "(this call raised a TypeError before the fix)"
)

print()
print("=== Item cold-start actually reachable via recommend(): register_new_item() ===")
# Previously score_new_item() only ever produced an isolated similarity number -- the new
# book could never actually appear in a real recommend() call, because query_topk() only
# searches the static NearestNeighbors index built once in build_index(). register_new_item()
# fixes that by scoring pending items directly against the profile and merging them into
# the candidate pool inside recommend() (see the change there).
NEW_BOOK_ITEM_IDX = -1  # synthetic id, guaranteed not to collide with a real item_idx (all >= 0)
svd_rec.register_new_item(NEW_BOOK_ITEM_IDX, new_book_fields)
print("user 1's top-5 recommendations after registering the new book:")
for r in svd_rec.recommend(1, k=5):
    flag = " <-- the new book" if r["item_idx"] == NEW_BOOK_ITEM_IDX else ""
    print(f"    [{r['reason']}] sim={r['similarity']:.3f} {r['title'][:55]}{flag}")

print()
print("=== User cold-start: onboarding-categories vs the old flat popularity fallback ===")
cold_builder = ColdStartProfileBuilder(
    items_df, svd_rec.vectorizer.item_matrix, svd_rec.item_idx_to_row, is_sparse=svd_rec.is_sparse,
)
svd_rec.attach_cold_start(cold_builder)

for label, cats in [("declares Mystery", ["Mystery"]), ("declares Romance", ["Romance"]), ("no info given at signup", [])]:
    print(f"-- new user, {label} --")
    for r in svd_rec.recommend_for_new_user(preferred_categories=cats, k=3):
        print(f"    [{r['reason']}] sim={r['similarity']:.3f} {r['title'][:55]}")


=== Item cold-start: scoring a brand-new book with zero interactions ===
cosine similarity of the brand-new book to user 1's profile: 0.515
(computed with zero retraining and zero interactions for the new book)

=== Item cold-start compatibility check: Weighted-TFIDF family ===
Weighted-TFIDF score_new_item() succeeded, vector shape=(1, 32768) (this call raised a TypeError before the fix)

=== Item cold-start actually reachable via recommend(): register_new_item() ===
user 1's top-5 recommendations after registering the new book:
    [content-similar] sim=0.847 In A Bind: A Private Investigator Crime and Suspense My
    [content-similar] sim=0.785 The Big Steal (The Reed Ferguson Mystery Series)
    [content-similar] sim=0.841 This Doesn't Happen In The Movies (The Reed Ferguson My
    [content-similar] sim=0.775 Bacon, Bodyguards, and Ballistics: A Piper Sandstone Cu
    [content-similar] sim=0.814 Loose Ends: A Private Investigator Crime and Suspense M

=== User cold-start: onboardin

### Simulated cold-start evaluation

The dataset has no real cold users to test against, so this pretends a sample of users have
zero training history and checks whether `onboarding-categories` beats the flat
`global-popularity` fallback on their real held-out validation positive. `proxy_categories`
(the user's top categories from real train history) stands in for what a genuinely new user
would type at signup -- it is only usable for *evaluating* this path offline; a real new user
would supply the categories directly, not have them read off hidden history.


In [34]:
def build_proxy_categories(items_indexed, interacted_items, user_idx, top_n=2):
    from collections import Counter
    seen = interacted_items.get(user_idx, set())
    cats = []
    for i in seen:
        if i in items_indexed.index:
            cats.extend(c.strip() for c in items_indexed.loc[i, "categories"].split(";") if c.strip())
    if not cats:
        return []
    return [c for c, _ in Counter(cats).most_common(top_n)]


def evaluate_simulated_cold_start(cold_builder, items_indexed, interacted_items,
                                   positives_df, negatives_dict, k, seed):
    popularity_ranks, onboarding_ranks = [], []
    for row in positives_df.itertuples():
        user_idx, positive = int(row.user_idx), int(row.pos_idx)
        candidates = [positive, *negatives_dict[user_idx]]

        pop_scores = items_indexed.loc[candidates, "popularity_score"].to_numpy()
        popularity_ranks.append(rank_from_scores(user_idx, candidates, pop_scores, seed))

        proxy_categories = build_proxy_categories(items_indexed, interacted_items, user_idx)
        profile_vec = cold_builder.onboarding_profile(proxy_categories) if proxy_categories else None
        if profile_vec is None:
            onboarding_ranks.append(popularity_ranks[-1])
            continue

        rows = [cold_builder.item_idx_to_row[i] for i in candidates]
        cand_matrix = cold_builder.item_matrix[rows]
        scores = (
            (cand_matrix @ profile_vec.T).toarray().ravel()
            if cold_builder.is_sparse else cand_matrix @ profile_vec.ravel()
        )
        onboarding_ranks.append(rank_from_scores(user_idx, candidates, scores, seed))

    return pd.DataFrame([
        summarise_ranks(popularity_ranks, k, "cold-start: global-popularity", "validation"),
        summarise_ranks(onboarding_ranks, k, "cold-start: onboarding-categories", "validation"),
    ])


cold_start_eval_df = evaluate_simulated_cold_start(
    cold_builder,
    svd_rec.items_df,
    train_interacted_items,
    val_positives,
    val_negatives,
    CONFIG["eval_k"],
    CONFIG["evaluation_seed"],
)
display(cold_start_eval_df)


,model,split,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users
0,cold-start: global-popularity,validation,0.405645,0.405645,0.040565,0.905006,0.253269,0.224836,64867
1,cold-start: onboarding-categories,validation,0.305533,0.305533,0.030553,0.904015,0.163173,0.142017,64867


# Report checklist

Before writing this component up, make sure the report states:

- the four-rung complexity ladder (Popularity → Basic-TFIDF → SVD → SBERT) and the test-set Hit@10/NDCG@10/MRR for each rung, so the reader can see whether each added layer of complexity actually earned its keep over the rung below it;
- that Basic and Variant 1 differ in more than "SVD or not" — feature construction (hashing vs vocabulary), dimensionality, and hash-collision noise all differ too (see the "对比口径说明" note above); don't describe Variant 1 as "Basic + SVD";
- that none of these methods involve gradient-based training — TF-IDF/hashing and TruncatedSVD are deterministic given a fixed `random_state`, and Sentence-BERT uses frozen pretrained weights — so there is no seed-stability or epoch-selection experiment to report here, unlike the team's sequential/CF methods;
- the sampled-evaluation caveat: Hit@10/NDCG@10 are measured against 1 positive + 100 fixed negatives, not full-catalogue ranking, and `Precision@10`'s ceiling is `0.1` by construction (leave-one-out), not a sign of low accuracy;
- which hyperparameters were selected on validation only (`svd_n_components`, `verified_purchase_multiplier`, `include_author`), and confirm each was evaluated on test exactly once after selection;
- the `description` missing-value rate printed by the data-loading cell, and how it was handled (title/categories/features still populate `content_text` even when `description` is empty);
- SBERT's double-truncation caveat (1500-character truncation for TF-IDF/SVD vs. the model's own ~256-token limit) as a plausible reason SBERT does not outperform TF-IDF/SVD despite using richer sentence embeddings;
- `cold_start_users=0` in this dataset, so the `"popular"` reason in sample recommendations is almost entirely the `candidate_pool` fallback, not the cold-start branch — report the fallback-rate-vs-`candidate_pool` table as evidence;
- this dataset has `cold_start_users=0` (5-core filtered), so the cold-start section results are a simulated evaluation, not a measurement on real cold users -- state that distinction explicitly rather than reporting the numbers as if they were organic;
- `CandidateAccuracy@10` and `outputs/candidate_set_results.csv` follow the exact schema of `Kindle_SASRec_Sequential_Recommender.ipynb`'s own results table -- state this is a reporting/merge-compatibility choice, not a new evaluation metric;
- `content_text` was changed from an implicit `title*2 + categories*2` baseline to a genuinely unweighted one (`build_unweighted_content_text()`), adopted from `content_based_three_variants_weighted_tfidf_aligned.ipynb`; **Weighted-TFIDF** is the new variant that isolates the field-weighting question properly (see the field-weight search ablation) -- state that any numbers computed before this change are not directly comparable to numbers computed after it;
- `recommend()` now MMR-reranks its top-10 output by default (`diversify=True`); this changes the qualitative sample recommendations and the catalogue-diagnostics numbers (Catalog Coverage / Intra-list Diversity) but not Hit@10/NDCG@10/MRR, since those are computed by `score_candidates()` on the fixed candidate set, which MMR never touches -- state this distinction explicitly if reporting both sets of numbers together;
- the Weighted-TFIDF field-weight search now also uses bigrams (`ngram_range=(1, 2)`) and `sublinear_tf=True`; Basic-TFIDF was deliberately left on plain unigrams/linear tf as the simple reference point, so the two are no longer differing *only* in field weights -- note this alongside the existing "对比口径说明" caveat;
- SBERT向量现在会缓存到 `outputs/sbert_cache/`（按模型名+content_text指纹命名），重跑notebook不用重新等mpnet那~20分钟的encode；content_text一变指纹跟着变，缓存自动失效，不会用旧向量算分。
- added a `Weighted-SVD-TUNED` family (`FieldWeightedSVDContentVectorizer`, ablation 11): SVD on top of the already field-weighted counts, to check whether dimensionality reduction still hurts once applied to the improved representation instead of plain TF-IDF; picked via validation NDCG@10 like every other hyperparameter, and included as a sixth competing family in the FINAL LOCK-AND-TEST cell;
- raised `diagnostic_n_users` from 50 to 5000: at 50 users the theoretical max Catalog Coverage was ~0.006 (500 recommended slots / 81322 items), and the observed 0.006 was ~97% of that ceiling -- the number reflected sample size, not real cross-user redundancy; 5000 users gives a ceiling of ~0.615, so the metric now has room to actually distinguish diverse from redundant recommendations;
- added `recommend_as_dataframe()`: adapts `recommend()`'s native list-of-dict output into the same rank/item_idx/score(+metadata) DataFrame shape as the sequential notebook's `recommend_top_k()`, without changing `recommend()` itself (several existing demo/ablation cells already depend on the list-of-dict format);
- `evaluate_catalogue_diagnostics()` now returns `(per_user_df, aggregate_dict)` with field names matching the sequential notebook's `recommendation_diagnostics()` exactly (`novelty`/`intra_list_diversity`/`avg_train_popularity` per user; `users`/`catalogue_coverage`/`unique_recommended_items` aggregated) -- previously it returned one already-averaged dict with mismatched Title-Case keys (`Catalog Coverage` vs the sequential notebook's `catalogue_coverage`) and discarded the per-user rows; results now also save to `outputs/content_based_catalogue_diagnostics.csv` / `outputs/content_based_catalogue_metrics.json`, mirroring the sequential notebook's two-file convention;
- fixed a real item-cold-start bug: `score_new_item()` used to always pass a flat text string to `vectorizer.transform_new_items()`, which crashed for `FieldWeightedTFIDFVectorizer`/`FieldWeightedSVDContentVectorizer` (i.e. the winning `Weighted-TFIDF-TUNED`/`Weighted-SVD-TUNED` families) since those need a DataFrame of raw per-field columns to apply field weights correctly; `score_new_item()` now takes a per-field dict and dispatches the correct input shape per vectorizer type;
- added `register_new_item()`: previously a scored new item could never actually appear in a real `recommend()` call (the ANN index is a static snapshot from `build_index()`); registered items are now scored directly against each user's profile and merged into `recommend()`'s candidate pool before MMR/top-k selection -- serving-time only, does not touch `score_candidates()` or any NDCG@10/Hit@10/MRR number;
- added a Colab bootstrap cell (right after "Setup and Configuration"): no-op off Colab, mounts Drive and adds Drive-relative paths to `CONFIG["data_dir_candidates"]` on Colab -- adjust `COLAB_DATA_DIR_CANDIDATES` if your `processed_kindle/` folder lives somewhere else in Drive;
- added an auto-install guard cell before `SBERTContentVectorizer` for `sentence-transformers` (only runs pip install if `CONFIG["run_sbert"]` is `True` and the import fails), so a fresh Colab runtime doesn't hit `ModuleNotFoundError`; `run_sbert` stays `True` by default (unchanged, so existing SBERT-MiniLM/mpnet comparison rows in the results table are unaffected) -- flip it to `False` if you want the rest of the notebook to run offline without the dependency.
- added a "Model Handoff" section at the very end: saves `content_based_final_vectorizer.joblib` (fitted vectorizer, for scoring brand-new items -- requires this notebook's classes to already be defined, like `gru4rec_best.pt` requires `GRU4Rec`), `content_based_item_embeddings.npz`/`.npy` + `content_based_item_idx_order.npy` (plain arrays, no custom classes needed -- for a hybrid-fusion notebook that only wants embeddings), and `content_based_model_card.json` (which model won, its hyperparameters, and load instructions). The pretrained SBERT model and the fitted NearestNeighbors index are both deliberately excluded from the serialised vectorizer to keep the checkpoint small.
- also exports `content_based_candidate_scores.csv.gz`: raw `user_idx, item_idx, split, score, is_positive` rows for every official validation/test candidate (~13M rows, gzip-compressed) -- the piece an actual score-level hybrid fusion needs, so a teammate can align by user_idx/item_idx against her own model's scores and blend, without touching this notebook's classes or reconstructing user profiles herself.
- on Colab, `OUTPUT_DIR`/`SHARED_OUTPUTS_DIR` are rooted under `COLAB_OUTPUT_ROOT` (a Drive folder, set in the Colab bootstrap cell) instead of the ephemeral runtime disk, so every artifact this notebook writes survives a runtime restart; off Colab both paths are byte-for-byte unchanged from before.
- added a core-dependency check right at the top (before the Colab bootstrap): pip-installs `numpy`/`pandas`/`scipy`/`scikit-learn`/`joblib` only if missing, so this notebook runs the same way on a fresh local machine as it does on Colab (which already has them preinstalled, so this is a no-op there).
- `audit_protocol()` passed with no failures before any number below it was collected — this is the leakage-and-alignment guarantee behind every metric in this notebook, worth one sentence in the report's methodology section.

Limitations to state explicitly: offline sampled metrics do not measure full-catalogue retrieval or post-deployment behaviour; `verified_purchase_multiplier` was chosen from a small validation-set grid search (1.0 / 1.2 / 1.5 / 2.0) on one dataset and may not generalise; Sentence-BERT was only evaluated when internet access was available and was not tuned beyond the two named pretrained checkpoints.

In [35]:
# Retrieval diagnostic for the same deterministic diagnostic-user sample.
train_lengths = [
    len(recommenders["ContentBased-SVD"].interacted_items.get(int(user), set()))
    for user in sample_users_for_pool_test
]
print(
    "training interactions min/median/mean/max:",
    np.min(train_lengths),
    np.median(train_lengths),
    np.mean(train_lengths),
    np.max(train_lengths),
)
print(
    "candidate_pool=5:",
    measure_fallback_rate(
        recommenders["ContentBased-SVD"],
        sample_users_for_pool_test,
        k=10,
        candidate_pool=5,
    ),
)


training interactions min/median/mean/max: 3 6.0 12.8416 556
candidate_pool=5: {'candidate_pool': 5, 'avg_popular_items_per_user': 5.2064, 'pct_users_with_any_fallback': 1.0, 'pct_recs_that_are_fallback': 0.52064}


In [36]:
# FINAL LOCK-AND-TEST CELL
#
# This is the only cell in the aligned notebook that evaluates test. Run it only
# after every validation-only cell above has finished and the configuration is locked.

CONFIG["svd_n_components"] = BEST_SVD_DIM
CONFIG["verified_purchase_multiplier"] = BEST_VERIFIED_MULTIPLIER

tuned_svd_items_df = items_df.copy()
tuned_svd_items_df["content_text"] = build_content_text(
    tuned_svd_items_df,
    CONFIG["description_truncate_chars"],
    include_author=BEST_INCLUDE_AUTHOR,
)

# First validate the combined tuned-SVD configuration. Individual ablation winners
# can interact, so the combined configuration must be checked before architecture selection.
tuned_svd_val_recommender, tuned_svd_val_result = run_variant(
    "TFIDF-SVD-TUNED",
    SVDContentVectorizer(
        CONFIG["svd_max_features"],
        CONFIG["svd_min_df"],
        BEST_SVD_DIM,
    ),
    tuned_svd_items_df,
    train_interactions,
    n_users,
    n_items,
    use_lazy_profiles=False,
    eval_splits=("validation",),
    verified_purchase_multiplier=BEST_VERIFIED_MULTIPLIER,
)

# Also validate the selected Weighted-TFIDF configuration (chosen on validation NDCG@10
# in the field-weight search ablation above) before it enters the family comparison.
weighted_tfidf_val_recommender, weighted_tfidf_val_result = run_variant(
    "Weighted-TFIDF-TUNED",
    FieldWeightedTFIDFVectorizer(
        CONFIG["basic_n_features"],
        BEST_WEIGHTED_FIELD_WEIGHTS,
        CONFIG["description_truncate_chars"],
        ngram_range=CONFIG["weighted_tfidf_ngram_range"],
        sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
    ),
    items_df,
    train_interactions,
    n_users,
    n_items,
    use_lazy_profiles=True,
    eval_splits=("validation",),
)

# Also validate Field-Weighted TF-IDF + SVD (chosen dim from the ablation above) --
# a fifth family, testing whether SVD helps once stacked on the already field-weighted
# representation instead of plain TF-IDF.
weighted_svd_val_recommender, weighted_svd_val_result = run_variant(
    "Weighted-SVD-TUNED",
    FieldWeightedSVDContentVectorizer(
        CONFIG["basic_n_features"],
        BEST_WEIGHTED_FIELD_WEIGHTS,
        CONFIG["description_truncate_chars"],
        BEST_WEIGHTED_SVD_DIM,
        ngram_range=CONFIG["weighted_tfidf_ngram_range"],
        sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
    ),
    items_df,
    train_interactions,
    n_users,
    n_items,
    use_lazy_profiles=False,
    eval_splits=("validation",),
)

# Compare pre-declared content model families using validation NDCG@10 only.
family_validation_rows = results_df[
    (results_df["split"] == "validation")
    & (results_df["model"].isin([
        "Basic-TFIDF",
        "ContentBased-SBERT-MiniLM",
        "ContentBased-SBERT-mpnet",
    ]))
].copy()
family_validation_rows = pd.concat(
    [
        family_validation_rows,
        pd.DataFrame(tuned_svd_val_result),
        pd.DataFrame(weighted_tfidf_val_result),
        pd.DataFrame(weighted_svd_val_result),
    ],
    ignore_index=True,
)
best_family_row = family_validation_rows.loc[
    family_validation_rows[CONFIG["primary_metric"]].idxmax()
]
SELECTED_CONTENT_MODEL = str(best_family_row["model"])

print(
    f"Locked final content model={SELECTED_CONTENT_MODEL} using validation "
    f"{CONFIG['primary_metric']}={best_family_row[CONFIG['primary_metric']]:.6f}"
)
display(
    family_validation_rows.sort_values(
        CONFIG["primary_metric"],
        ascending=False,
    )
)

if SELECTED_CONTENT_MODEL == "Basic-TFIDF":
    final_vectorizer = TFIDFHashingVectorizer(CONFIG["basic_n_features"])
    final_items_df = items_df
    final_use_lazy_profiles = True
    final_verified_multiplier = 1.0
elif SELECTED_CONTENT_MODEL == "Weighted-TFIDF-TUNED":
    final_vectorizer = FieldWeightedTFIDFVectorizer(
        CONFIG["basic_n_features"],
        BEST_WEIGHTED_FIELD_WEIGHTS,
        CONFIG["description_truncate_chars"],
        ngram_range=CONFIG["weighted_tfidf_ngram_range"],
        sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
    )
    final_items_df = items_df
    final_use_lazy_profiles = True
    final_verified_multiplier = 1.0
elif SELECTED_CONTENT_MODEL == "Weighted-SVD-TUNED":
    final_vectorizer = FieldWeightedSVDContentVectorizer(
        CONFIG["basic_n_features"],
        BEST_WEIGHTED_FIELD_WEIGHTS,
        CONFIG["description_truncate_chars"],
        BEST_WEIGHTED_SVD_DIM,
        ngram_range=CONFIG["weighted_tfidf_ngram_range"],
        sublinear_tf=CONFIG["weighted_tfidf_sublinear_tf"],
    )
    final_items_df = items_df
    final_use_lazy_profiles = False
    final_verified_multiplier = 1.0
elif SELECTED_CONTENT_MODEL == "TFIDF-SVD-TUNED":
    final_vectorizer = SVDContentVectorizer(
        CONFIG["svd_max_features"],
        CONFIG["svd_min_df"],
        BEST_SVD_DIM,
    )
    final_items_df = tuned_svd_items_df
    final_use_lazy_profiles = False
    final_verified_multiplier = BEST_VERIFIED_MULTIPLIER
elif SELECTED_CONTENT_MODEL == "ContentBased-SBERT-MiniLM":
    final_vectorizer = SBERTContentVectorizer(
        CONFIG["sbert_model_name"],
        CONFIG["sbert_batch_size"],
    )
    final_items_df = items_df
    final_use_lazy_profiles = False
    final_verified_multiplier = 1.0
elif SELECTED_CONTENT_MODEL == "ContentBased-SBERT-mpnet":
    final_vectorizer = SBERTContentVectorizer(
        "all-mpnet-base-v2",
        CONFIG["sbert_batch_size"],
    )
    final_items_df = items_df
    final_use_lazy_profiles = False
    final_verified_multiplier = 1.0
else:
    raise ValueError(f"Unsupported selected model: {SELECTED_CONTENT_MODEL}")

# run_variant("test") automatically builds profiles from train + validation.
final_recommender, final_test_results = run_variant(
    f"{SELECTED_CONTENT_MODEL}-FINAL",
    final_vectorizer,
    final_items_df,
    train_interactions,
    n_users,
    n_items,
    use_lazy_profiles=final_use_lazy_profiles,
    eval_splits=("test",),
    verified_purchase_multiplier=final_verified_multiplier,
)

diagnostic_rng = np.random.default_rng(CONFIG["evaluation_seed"])
diagnostic_users = diagnostic_rng.choice(
    np.arange(n_users),
    size=min(CONFIG["diagnostic_n_users"], n_users),
    replace=False,
)
# evaluate_catalogue_diagnostics() now returns (per_user_df, aggregate_dict), matching
# the sequential notebook's recommendation_diagnostics() shape and field names exactly
# (novelty/intra_list_diversity/avg_train_popularity per user; users/catalogue_coverage/
# unique_recommended_items aggregated).
final_catalogue_diagnostics_df, final_catalogue_metrics = evaluate_catalogue_diagnostics(
    final_recommender,
    diagnostic_users,
    n_items,
    CONFIG["top_k"],
    train_item_counts,
)
final_catalogue_metrics["model"] = f"{SELECTED_CONTENT_MODEL}-FINAL"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Cross-team schema parity: same file/columns the sequential notebook writes for its own
# locked test row, so the final hybrid-fusion comparison can read one CSV per teammate.
save_candidate_set_results(to_report_names(pd.DataFrame(final_test_results)))

# Cross-team schema parity for catalogue diagnostics too: same two-file convention as
# the sequential notebook's gru4rec_catalogue_diagnostics.csv / gru4rec_catalogue_metrics.json
# (per-user rows in a CSV, aggregate numbers in a JSON), just named for this model family.
final_catalogue_diagnostics_df.to_csv(
    OUTPUT_DIR / "content_based_catalogue_diagnostics.csv", index=False
)
with open(OUTPUT_DIR / "content_based_catalogue_metrics.json", "w", encoding="utf-8") as handle:
    json.dump(final_catalogue_metrics, handle, indent=2)

with open(OUTPUT_DIR / "final_locked_test_results.json", "w", encoding="utf-8") as handle:
    json.dump(
        {
            "selected_by": CONFIG["primary_metric"],
            "selected_model": SELECTED_CONTENT_MODEL,
            "validation_candidates": family_validation_rows.to_dict(orient="records"),
            "locked_configuration": {
                "svd_dim": BEST_SVD_DIM,
                "weighted_svd_dim": BEST_WEIGHTED_SVD_DIM,
                "include_author": BEST_INCLUDE_AUTHOR,
                "verified_purchase_multiplier": BEST_VERIFIED_MULTIPLIER,
                "model_seed": CONFIG["model_seed"],
                "evaluation_seed": CONFIG["evaluation_seed"],
                "validation_context": "train",
                "test_context": "train+validation",
            },
            "test_results": final_test_results,
            "catalogue_diagnostics": final_catalogue_metrics,
        },
        handle,
        indent=2,
    )

print("Final test result (1 positive + 100 fixed negatives):")
display(pd.DataFrame(final_test_results))
print("Full-catalogue diagnostics on the shared deterministic user sample:")
display(pd.DataFrame([final_catalogue_metrics]))
print("Per-user catalogue diagnostics (first 5 rows):")
display(final_catalogue_diagnostics_df.head())


[TFIDF-SVD-TUNED] item_matrix shape=(81322, 256)                                                    
[TFIDF-SVD-TUNED] validation: Hit@10=0.5895 NDCG@10=0.3974 MRR=0.3541                               
[Weighted-TFIDF-TUNED] item_matrix shape=(81322, 32768)                                             
[Weighted-TFIDF-TUNED] validation: Hit@10=0.5932 NDCG@10=0.4341 MRR=0.4006                          
[Weighted-SVD-TUNED] item_matrix shape=(81322, 512)                                                 
[Weighted-SVD-TUNED] validation: Hit@10=0.6065 NDCG@10=0.4141 MRR=0.3702                            
Locked final content model=Weighted-TFIDF-TUNED using validation NDCG@10=0.434093


,model,split,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users
4,Weighted-TFIDF-TUNED,validation,0.593152,0.593152,0.059315,0.906863,0.434093,0.400600,64867
5,Weighted-SVD-TUNED,validation,0.606503,0.606503,0.060650,0.906995,0.414140,0.370217,64867
3,TFIDF-SVD-TUNED,validation,0.589499,0.589499,0.058950,0.906827,0.397368,0.354133,64867
0,Basic-TFIDF,validation,0.537484,0.537484,0.053748,0.906312,0.389874,0.361362,64867
2,ContentBased-SBERT-mpnet,validation,0.478286,0.478286,0.047829,0.905726,0.308409,0.275441,64867
1,ContentBased-SBERT-MiniLM,validation,0.440101,0.440101,0.044010,0.905348,0.280078,0.251414,64867


[Weighted-TFIDF-TUNED-FINAL] item_matrix shape=(81322, 32768)                                       
[Weighted-TFIDF-TUNED-FINAL] test: Hit@10=0.5914 NDCG@10=0.4283 MRR=0.3935                          
Final test result (1 positive + 100 fixed negatives):


,model,split,Hit@10,Recall@10,Precision@10,CandidateAccuracy@10,NDCG@10,MRR,users
0,Weighted-TFIDF-TUNED-FINAL,test,0.591441,0.591441,0.059144,0.906846,0.428251,0.393531,64867


Full-catalogue diagnostics on the shared deterministic user sample:


,users,catalogue_coverage,unique_recommended_items,model
0,5000,0.248137,20179,Weighted-TFIDF-TUNED-FINAL


Per-user catalogue diagnostics (first 5 rows):


,user_idx,novelty,intra_list_diversity,avg_train_popularity
0,39754,15.563100,0.755556,85.0
1,15519,15.675596,0.688889,26.4
2,26537,16.808202,0.044444,8.1
3,17169,14.543150,0.000000,111.1
4,60483,15.695243,0.155556,20.2


# Model Handoff (for teammates / hybrid fusion)

Everything above evaluates and locks a model; nothing so far actually **saves** it in a
form anyone else can reuse without re-running this whole notebook -- the sequential
notebook's `gru4rec_best.pt` plays that role for GRU4Rec (a saved checkpoint that skips
re-training), and content-based had no equivalent.

This section saves three files to `outputs/`, right after `FINAL LOCK-AND-TEST CELL` has
produced `final_vectorizer`/`SELECTED_CONTENT_MODEL`:

1. **`content_based_final_vectorizer.joblib`** -- the fitted vectorizer object itself
   (idf weights / SVD components / field weights, whichever the winning model used).
   Needed to score a *brand-new* item's raw text via `transform_new_items()` without
   refitting on the full catalogue. Like `gru4rec_best.pt`, loading this requires this
   notebook's vectorizer classes (`FieldWeightedTFIDFVectorizer` etc.) to already be
   defined in whatever session loads it -- a plain `joblib.load()` from a fresh kernel
   with no class definitions will fail with an `AttributeError`.
2. **`content_based_item_embeddings.npz` (sparse) or `.npy` (dense)** -- just the raw
   item vectors, as plain arrays. No custom classes needed to read this one -- a
   hybrid-fusion notebook that only wants to combine embeddings (not score new items)
   can load it with plain `numpy`/`scipy`, with zero dependency on this notebook's code.
3. **`content_based_model_card.json`** -- which model won, its hyperparameters, file
   names, and short load/usage snippets for both artifacts above.

To keep the checkpoint lean: the pretrained SBERT model itself (hundreds of MB) is
deliberately *not* serialised if SBERT wins -- it is better re-downloaded by name than
duplicated in a file -- and the fitted `NearestNeighbors` index is also excluded, since
rebuilding it from the saved embeddings via `build_index()` is cheap and it otherwise
just duplicates a second copy of the same item vectors internally.

In [37]:
import joblib  # standard scikit-learn dependency -- no extra install needed

MODEL_HANDOFF_DIR = OUTPUT_DIR  # reuse the same outputs/ directory as every other artifact


def save_content_based_handoff(vectorizer, model_name: str, locked_configuration: dict) -> dict:
    """Serialises the final fitted vectorizer + its item embeddings so a teammate (or a
    hybrid-fusion notebook) can reuse this model without re-fitting on the full item
    catalogue. See the markdown note above for why there are two separate artifacts and
    what each one needs (or doesn't need) at load time."""
    vectorizer_path = MODEL_HANDOFF_DIR / "content_based_final_vectorizer.joblib"

    # Detach the two large/redundant pieces before dumping, then restore them on this
    # live object -- final_recommender (built from this same vectorizer) keeps working
    # normally for the rest of this session; only the *serialised copy* omits them.
    detached_sbert_model = None
    if isinstance(vectorizer, SBERTContentVectorizer):
        detached_sbert_model, vectorizer.model = vectorizer.model, None
    detached_nn_index, vectorizer.nn_index = vectorizer.nn_index, None
    try:
        joblib.dump(vectorizer, vectorizer_path)
    finally:
        vectorizer.nn_index = detached_nn_index
        if detached_sbert_model is not None:
            vectorizer.model = detached_sbert_model

    item_matrix = vectorizer.item_matrix
    is_sparse = sp.issparse(item_matrix)
    if is_sparse:
        embeddings_path = MODEL_HANDOFF_DIR / "content_based_item_embeddings.npz"
        sp.save_npz(embeddings_path, item_matrix.tocsr())
    else:
        embeddings_path = MODEL_HANDOFF_DIR / "content_based_item_embeddings.npy"
        np.save(embeddings_path, np.asarray(item_matrix))

    item_idx_path = MODEL_HANDOFF_DIR / "content_based_item_idx_order.npy"
    np.save(item_idx_path, np.asarray(vectorizer.item_idx_order))

    model_card = {
        "selected_model": model_name,
        "embedding_dim": int(item_matrix.shape[1]),
        "n_items": int(item_matrix.shape[0]),
        "is_sparse": bool(is_sparse),
        "similarity_metric": "cosine (vectors are L2-normalised, so dot product == cosine similarity)",
        "vectorizer_file": vectorizer_path.name,
        "embeddings_file": embeddings_path.name,
        "item_idx_file": item_idx_path.name,
        "locked_configuration": locked_configuration,
        "how_to_use": {
            "score_new_item_text": (
                "vectorizer = joblib.load('content_based_final_vectorizer.joblib'); "
                "requires this notebook's vectorizer classes to already be defined; call "
                "vectorizer.build_index(n_neighbors=...) once before query_topk() (the "
                "fitted index was deliberately not saved -- see markdown note); then "
                "vectorizer.transform_new_items(...) to embed a brand-new item's text."
            ),
            "reuse_embeddings_only": (
                "item_matrix = scipy.sparse.load_npz(...) if is_sparse else np.load(...); "
                "item_idx_order = np.load('content_based_item_idx_order.npy'); "
                "row = {idx: i for i, idx in enumerate(item_idx_order)}[item_idx]; "
                "score = item_matrix[row] @ user_profile_vector -- no custom classes needed."
            ),
        },
    }
    model_card_path = MODEL_HANDOFF_DIR / "content_based_model_card.json"
    with open(model_card_path, "w", encoding="utf-8") as handle:
        json.dump(model_card, handle, indent=2)

    return model_card


final_model_card = save_content_based_handoff(
    final_vectorizer,
    SELECTED_CONTENT_MODEL,
    {
        "svd_dim": BEST_SVD_DIM,
        "weighted_svd_dim": BEST_WEIGHTED_SVD_DIM,
        "include_author": BEST_INCLUDE_AUTHOR,
        "verified_purchase_multiplier": BEST_VERIFIED_MULTIPLIER,
        "field_weights": (
            BEST_WEIGHTED_FIELD_WEIGHTS
            if SELECTED_CONTENT_MODEL in {"Weighted-TFIDF-TUNED", "Weighted-SVD-TUNED"}
            else None
        ),
    },
)

print("Saved content-based handoff artifacts to", OUTPUT_DIR.resolve())
for key in ("vectorizer_file", "embeddings_file", "item_idx_file"):
    print(f"  outputs/{final_model_card[key]}")
print("  outputs/content_based_model_card.json")


Saved content-based handoff artifacts to /Users/lumi/PycharmProjects/PythonProject/comp9727/teamProject/content_based_results_ALIGNED
  outputs/content_based_final_vectorizer.joblib
  outputs/content_based_item_embeddings.npz
  outputs/content_based_item_idx_order.npy
  outputs/content_based_model_card.json


### Per-candidate score export (for hybrid score-level fusion)

Everything above hands off the *model* (fitted vectorizer + embeddings) -- a teammate
who wants to actually blend content-based scores with her own CF/sequential scores
(the same way the sequential notebook blends GRU4Rec and Markov-1 via
`standardise_candidate_scores()`) would otherwise have to reload this notebook's classes
and rebuild user profiles from `interactions_clean.csv` herself just to get scores.

This cell instead exports the raw score already computed for every official candidate
(the same 1-positive+100-negative sets `evaluate_ranking()` scores, on both validation
and test) to `outputs/content_based_candidate_scores.csv.gz`: plain
`user_idx, item_idx, split, score, is_positive` rows -- no custom classes, no vectorizer,
no user-profile logic needed to consume this file. A hybrid notebook only has to align
rows by `user_idx`/`item_idx` against her own model's scores over the same candidate
sets, then z-score and blend (per user, across that user's 101 candidates) however she
likes -- `standardise_candidate_scores()` above is the same formula if she wants to reuse
it verbatim.

Gzip-compressed because this is long-format (~64,867 users x 101 candidates x 2 splits =
~13M rows) -- keeps the file to a reasonable size without adding a parquet dependency.

In [38]:
def export_candidate_scores(
    recommender,
    positives_df: pd.DataFrame,
    negatives_dict: dict[int, list[int]],
    split: str,
) -> pd.DataFrame:
    """Raw per-(user, candidate) scores for the official 1-positive+100-negative protocol
    -- the same candidate sets evaluate_ranking() scores, just kept instead of being
    collapsed into a summary metric. See the markdown note above for why this is the
    piece a hybrid-fusion notebook actually needs for score-level ensembling."""
    split = normalise_split_name(split)
    rows = []
    for row in positives_df.itertuples():
        user_idx = int(row.user_idx)
        positive = int(row.pos_idx)
        candidates = [positive, *negatives_dict[user_idx]]
        scores = recommender.score_candidates(user_idx, candidates)
        rows.extend(
            {
                "user_idx": user_idx,
                "item_idx": int(item_idx),
                "split": split,
                "score": float(score),
                "is_positive": item_idx == positive,
            }
            for item_idx, score in zip(candidates, scores)
        )
    return pd.DataFrame(rows)


# The validation-context recommender for the winning model lives in a different place
# depending on which family was selected -- run_all_variants() already built the
# validation-context recommenders for Basic-TFIDF/SBERT in `recommenders`; the three
# *-TUNED families were validated separately in the FINAL LOCK-AND-TEST CELL above.
_validation_recommender_by_model = {
    "Basic-TFIDF": recommenders.get("Basic-TFIDF"),
    "ContentBased-SBERT-MiniLM": recommenders.get("ContentBased-SBERT-MiniLM"),
    "ContentBased-SBERT-mpnet": recommenders.get("ContentBased-SBERT-mpnet"),
    "TFIDF-SVD-TUNED": globals().get("tuned_svd_val_recommender"),
    "Weighted-TFIDF-TUNED": globals().get("weighted_tfidf_val_recommender"),
    "Weighted-SVD-TUNED": globals().get("weighted_svd_val_recommender"),
}
selected_validation_recommender = _validation_recommender_by_model[SELECTED_CONTENT_MODEL]
if selected_validation_recommender is None:
    raise RuntimeError(
        f"No validation-context recommender found for {SELECTED_CONTENT_MODEL!r} -- "
        "make sure the cells above (run_all_variants() and/or the FINAL LOCK-AND-TEST "
        "CELL) have already run in this session."
    )

validation_scores_df = export_candidate_scores(
    selected_validation_recommender,
    load_eval_positives(DATA_DIR, "validation"),
    load_eval_negatives(DATA_DIR, "validation"),
    "validation",
)
test_scores_df = export_candidate_scores(
    final_recommender,  # test-context recommender for the same winning model, from FINAL LOCK-AND-TEST CELL
    load_eval_positives(DATA_DIR, "test"),
    load_eval_negatives(DATA_DIR, "test"),
    "test",
)

candidate_scores_df = pd.concat([validation_scores_df, test_scores_df], ignore_index=True)
candidate_scores_path = OUTPUT_DIR / "content_based_candidate_scores.csv.gz"
candidate_scores_df.to_csv(candidate_scores_path, index=False, compression="gzip")

print(f"Exported {len(candidate_scores_df):,} (user, candidate) score rows for model={SELECTED_CONTENT_MODEL}")
print(f"  -> {candidate_scores_path}")
display(candidate_scores_df.head())


Exported 13,103,134 (user, candidate) score rows for model=Weighted-TFIDF-TUNED
  -> content_based_results_ALIGNED/content_based_candidate_scores.csv.gz


,user_idx,item_idx,split,score,is_positive
0,0,2617,validation,0.071234,True
1,0,5053,validation,0.036460,False
2,0,37497,validation,0.033347,False
3,0,18865,validation,0.026172,False
4,0,60625,validation,0.020687,False
